# Taller 3 — Reconocimiento de entidades clínicas con BERT preentrenado

**Carlos Javier Cepeda, David Salamanca, Jose Milciades Ordoñez**

Trabajaremos paso a paso sobre el mismo corpus SPACCC de los talleres 1 y 2. Tomamos como referencia el notebook de clase `1_text_classification_with_hf.ipynb`, que utiliza Hugging Face y el checkpoint español BETO (`dccuchile/bert-base-spanish-wwm-cased`).

El ejemplo de clase clasifica noticias completas. Nuestro ejercicio es **NER: clasificación por token**, por lo que en una etapa posterior usaremos una cabeza de clasificación por token y alinearemos las etiquetas BIO con los subtokens de BERT. Referencia: [guía oficial de clasificación por token de Hugging Face](https://huggingface.co/docs/transformers/tasks/token_classification).



### PASO 01 — Dependencias para descargar y leer los datos

Usamos `requests` para descargar los mismos archivos Parquet de Hugging Face que en los talleres anteriores y `pandas`/`pyarrow` para leerlos. Conservamos las fuentes originales para comprobar sus huellas y facilitar la comparación. Incorporaremos las herramientas de modelos de Hugging Face en la siguiente etapa.

In [1]:
import sys
import subprocess

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'pandas==2.3.3', 'pyarrow>=14', 'requests>=2.31',
                       "https://github.com/explosion/spacy-models/releases/download/"
                       "es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl"
                      ])
print('PASO 01 OK. Dependencias de datos disponibles.')



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 62.8 MB/s eta 0:00:00
PASO 01 OK. Dependencias de datos disponibles.


### PASO 02 — Rutas y fuentes

Descargamos dos tablas de anotaciones y una tabla de documentos completos:

| Recurso | Contenido |
|---|---|
| `IEETA/SPACCC-Spanish-NER`, train | Anotaciones de los 750 documentos oficiales de entrenamiento |
| `IEETA/SPACCC-Spanish-NER`, test | Anotaciones de los 250 documentos oficiales de prueba |
| `IEETA/SPACCC-documents`, train | Documentos completos de ambas particiones |

Una fila de anotaciones describe una mención, no un documento. La columna `text` contiene el fragmento anotado; `start_span` y `end_span` indican su ubicación en el documento. Para NER necesitaremos el texto completo, disponible en `df_docs['document']`.

Todavía no separamos validación: en el siguiente paso recuperaremos la partición fija de 600/150 documentos de los talleres anteriores. Test se descarga para verificar integridad y permanece fuera de la selección y entrenamiento.

In [2]:
from pathlib import Path
import hashlib
import json
import platform
import time

import pandas as pd
import pyarrow
import requests
from IPython.display import display

ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
WORK = ROOT / 'spaccc_bert'
CACHE = WORK / 'data'
CACHE.mkdir(parents=True, exist_ok=True)
LABELS = ['CHEMICAL', 'DISEASE', 'PROCEDURE', 'PROTEIN', 'SYMPTOM']
MODEL_CHECKPOINT = 'dccuchile/bert-base-spanish-wwm-cased'  # Referencia de clase; aún no se descarga.
SOURCES = {
    'annotations_train.parquet': ('IEETA/SPACCC-Spanish-NER', 'train'),
    'annotations_test.parquet': ('IEETA/SPACCC-Spanish-NER', 'test'),
    'documents.parquet': ('IEETA/SPACCC-documents', 'train'),
}
# Huellas de los archivos utilizados en los talleres 1 y 2.
EXPECTED_SHA256 = {
    'annotations_train.parquet': '76c692d7cc49d35d030aeb5c27523862ebd7585bdcf31e596f2f7d6ebe37c033',
    'annotations_test.parquet': '53bf36209410f657f773cfeb804b08487ab1e8d23e3af9e972cc3927140e1413',
    'documents.parquet': 'd866b787254d0ebdc4e7453ea3fdce25edfb1fc058b536404d1dcf15e5a40f24',
}
REPORT = {'stage': 'descarga_y_auditoria', 'model_checkpoint_planned': MODEL_CHECKPOINT,
    'versions': {'python': platform.python_version(), 'pandas': pd.__version__,
                 'pyarrow': pyarrow.__version__, 'requests': requests.__version__},
    'sources': {name: f'https://huggingface.co/datasets/{repo}/resolve/main/data/{split}-00000-of-00001.parquet'
                for name, (repo, split) in SOURCES.items()}}
print('PASO 02 OK. Carpeta de datos:', CACHE)

PASO 02 OK. Carpeta de datos: /kaggle/working/spaccc_bert/data


### PASO 03 — Descarga y auditoría del corpus

Reutilizamos los archivos si ya están descargados. Verificamos su lectura, las huellas SHA-256, las columnas, las categorías, la correspondencia entre anotaciones y documentos y la ausencia de textos duplicados entre train y test. Las dos diferencias de comillas conocidas del taller 1 se registran explícitamente; cualquier otra discrepancia detiene la auditoría.

Si las huellas cambian, detenemos la comparación: necesitamos revisar la fuente antes de mezclar versiones del corpus. No se normaliza ni modifica el texto original, para conservar las posiciones de las anotaciones.

In [3]:
# PASO 03 — Descarga, textos completos y auditoría de todas las anotaciones
def download_parquet(repo, split, name):
    """Descarga el parquet `split` del dataset `repo` (Hugging Face) y lo guarda como
    `name` dentro de la carpeta de caché. Si el archivo ya existe en caché lo reutiliza
    sin volver a descargarlo. Reintenta la descarga hasta 3 veces y valida que el
    parquet descargado se pueda leer antes de reemplazar la caché (para nunca dejar
    una respuesta corrupta o incompleta guardada como si fuera válida).
    Devuelve el contenido ya cargado como DataFrame de pandas."""
    path = CACHE / name
    if not path.exists():
        url = f'https://huggingface.co/datasets/{repo}/resolve/main/data/{split}-00000-of-00001.parquet'
        print('Descargando:', name, flush=True)
        error = None
        for attempt in range(3):
            try:
                response = requests.get(url, timeout=(15, 120))
                response.raise_for_status()
                tmp = path.with_suffix('.tmp')
                tmp.write_bytes(response.content)
                pd.read_parquet(tmp)  # No guardar una respuesta inválida como caché.
                tmp.replace(path)
                break
            except Exception as exc:
                error = exc
                print(f'Intento {attempt + 1}/3: {type(exc).__name__}: {exc}', flush=True)
        if not path.exists():
            raise RuntimeError('PASO 03: revisa Internet de Kaggle y comparte este error.') from error
    actual_sha = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual_sha == EXPECTED_SHA256[name], f'El archivo {name} difiere del corpus de los talleres anteriores.'
    return pd.read_parquet(path)

t0 = time.perf_counter()
df_train = download_parquet('IEETA/SPACCC-Spanish-NER', 'train', 'annotations_train.parquet')
df_test = download_parquet('IEETA/SPACCC-Spanish-NER', 'test', 'annotations_test.parquet')
df_docs = download_parquet('IEETA/SPACCC-documents', 'train', 'documents.parquet')
def document_id(value):
    """Obtiene el identificador de un documento a partir de su nombre de archivo:
    quita la extensión (p. ej. '.txt') y, si existe, el prefijo 'es-'."""
    return Path(str(value)).stem.removeprefix('es-')
required = {'filename', 'label', 'start_span', 'end_span', 'text'}
for frame in (df_train, df_test):
    assert required <= set(frame.columns), 'Columnas de anotación inesperadas.'
    assert not frame[list(required)].isnull().any().any(), 'Anotaciones con valores nulos.'
    frame['doc_id'] = frame.filename.map(document_id)
    assert set(frame.label) <= set(LABELS), 'Categorías nuevas: revisar TAGS.'
assert {'filename', 'document'} <= set(df_docs.columns), 'Columnas de documentos inesperadas.'
assert not df_docs[['filename', 'document']].isnull().any().any(), 'Documentos con valores nulos.'
df_docs['doc_id'] = df_docs.filename.map(document_id)
assert not df_docs.doc_id.duplicated().any(), 'Identificadores de documento duplicados.'
assert df_docs.document.map(lambda x: isinstance(x, str) and bool(x)).all()
texts = dict(zip(df_docs.doc_id, df_docs.document))
official_train_ids = sorted(df_train.doc_id.unique())
official_test_ids = sorted(df_test.doc_id.unique())
assert not set(official_train_ids) & set(official_test_ids), 'Fuga: documentos en train y test.'
assert (set(official_train_ids) | set(official_test_ids)) <= texts.keys(), 'Faltan documentos completos.'
# Auditar el esquema de test no implica usarlo para seleccionar el modelo.
bad_offsets, quote_only_differences = [], []
for split, frame in [('train', df_train), ('test', df_test)]:
    for row in frame.itertuples(index=False):
        a, b = int(row.start_span), int(row.end_span)
        actual = texts[row.doc_id][a:b]
        detail = {'split': split, 'doc_id': row.doc_id, 'start': a, 'end': b}
        if not (0 <= a < b <= len(texts[row.doc_id])):
            bad_offsets.append(detail)
        elif actual != row.text:
            if actual.replace(chr(34), '') == row.text.replace(chr(34), ''):
                quote_only_differences.append(detail)
            else:
                bad_offsets.append(detail)
assert not bad_offsets, f'Offsets incompatibles: {len(bad_offsets)}. Ejemplos: {bad_offsets[:5]}'
known_quotes = {('test', 'S0004-06142008000700003-1', 1560, 1647),
                ('test', 'S1134-80462005000100007-1', 127, 145)}
assert {(d['split'], d['doc_id'], d['start'], d['end']) for d in quote_only_differences} == known_quotes, 'Cambió la auditoría de comillas: revisar datos.'
# Huella de un texto normalizando espacios; nos sirve para detectar documentos
# duplicados o repetidos entre particiones (train/test) sin comparar cadenas completas.
fingerprint = lambda s: hashlib.sha256(' '.join(s.split()).encode()).hexdigest()
train_hashes = {fingerprint(texts[k]) for k in official_train_ids}
test_hashes = {fingerprint(texts[k]) for k in official_test_ids}
assert not train_hashes & test_hashes, 'Textos duplicados entre train y test: revisar partición.'
REPORT['data'] = {'train_documents': len(official_train_ids), 'test_documents': len(official_test_ids),
                  'train_annotations': len(df_train), 'test_annotations': len(df_test),
                  'offset_errors': len(bad_offsets), 'quote_only_differences': quote_only_differences,
                  'download_audit_seconds': time.perf_counter() - t0,
                  'sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                             for p in CACHE.glob('*.parquet')}}
assert len(official_train_ids) == 750 and len(official_test_ids) == 250
assert len(df_train) == 33757 and len(df_test) == 11239
REPORT['status'] = 'DATOS_DESCARGADOS_Y_VERIFICADOS'
REPORT['data']['matches_previous_workshops'] = True
REPORT['data']['documents_total'] = len(df_docs)
(REPORT_PATH := WORK / 'data_summary.json').write_text(json.dumps(REPORT, indent=2, ensure_ascii=False), encoding='utf-8')
from IPython.display import Markdown


def show_markdown_table(headers, rows):
    # Escapar separadores y saltos para conservar la estructura de la tabla.
    def cell(value):
        return str(value).replace('|', '&#124;').replace('\n', '<br>')
    lines = ['| ' + ' | '.join(map(cell, headers)) + ' |',
             '| ' + ' | '.join(['---'] * len(headers)) + ' |']
    lines.extend('| ' + ' | '.join(map(cell, row)) + ' |' for row in rows)
    display(Markdown('\n'.join(lines)))


display(Markdown('### PASO 03 OK — Datos descargados y verificados'))
show_markdown_table(['Partición', 'Documentos', 'Anotaciones'], [
    ['Entrenamiento oficial', len(official_train_ids), len(df_train)],
    ['Prueba oficial', len(official_test_ids), len(df_test)],
    ['Total', len(df_docs), len(df_train) + len(df_test)],
])

display(Markdown('#### Auditoría de integridad'))
show_markdown_table(['Verificación', 'Resultado'], [
    ['Errores de offsets', len(bad_offsets)],
    ['Diferencias conocidas de comillas', len(quote_only_differences)],
    ['Documentos compartidos entre train y test', len(set(official_train_ids) & set(official_test_ids))],
    ['Textos duplicados entre train y test', len(train_hashes & test_hashes)],
    ['Archivos idénticos a los talleres anteriores', 'Sí'],
    ['Tiempo de descarga y auditoría (segundos)', f"{REPORT['data']['download_audit_seconds']:.2f}"],
])

display(Markdown('#### Diferencias conocidas de comillas'))
show_markdown_table(['Partición', 'Documento', 'Inicio', 'Fin'], [
    [item['split'], item['doc_id'], item['start'], item['end']]
    for item in quote_only_differences
])

display(Markdown('#### Huellas de los archivos'))
show_markdown_table(['Archivo', 'SHA-256'], sorted(REPORT['data']['sha256'].items()))
display(Markdown(f'**Archivo de auditoría:** `{REPORT_PATH}`'))

### PASO 03 OK — Datos descargados y verificados

| Partición | Documentos | Anotaciones |
| --- | --- | --- |
| Entrenamiento oficial | 750 | 33757 |
| Prueba oficial | 250 | 11239 |
| Total | 1000 | 44996 |

#### Auditoría de integridad

| Verificación | Resultado |
| --- | --- |
| Errores de offsets | 0 |
| Diferencias conocidas de comillas | 2 |
| Documentos compartidos entre train y test | 0 |
| Textos duplicados entre train y test | 0 |
| Archivos idénticos a los talleres anteriores | Sí |
| Tiempo de descarga y auditoría (segundos) | 0.39 |

#### Diferencias conocidas de comillas

| Partición | Documento | Inicio | Fin |
| --- | --- | --- | --- |
| test | S0004-06142008000700003-1 | 1560 | 1647 |
| test | S1134-80462005000100007-1 | 127 | 145 |

#### Huellas de los archivos

| Archivo | SHA-256 |
| --- | --- |
| annotations_test.parquet | 53bf36209410f657f773cfeb804b08487ab1e8d23e3af9e972cc3927140e1413 |
| annotations_train.parquet | 76c692d7cc49d35d030aeb5c27523862ebd7585bdcf31e596f2f7d6ebe37c033 |
| documents.parquet | d866b787254d0ebdc4e7453ea3fdce25edfb1fc058b536404d1dcf15e5a40f24 |

**Archivo de auditoría:** `/kaggle/working/spaccc_bert/data_summary.json`

### Inspección de la carga

Revisamos dimensiones y una muestra de anotaciones de entrenamiento. Los conteos por categoría describen menciones anotadas; todavía no incluyen la etiqueta `O`, las etiquetas BIO ni las exclusiones por solapamiento, que se construirán más adelante.

In [4]:
overview = pd.DataFrame([
    {'recurso': 'Anotaciones train', 'filas': len(df_train), 'documentos': len(official_train_ids)},
    {'recurso': 'Anotaciones test', 'filas': len(df_test), 'documentos': len(official_test_ids)},
    {'recurso': 'Textos completos', 'filas': len(df_docs), 'documentos': df_docs.doc_id.nunique()},
])
display(overview)
display(df_train[['doc_id', 'text', 'label', 'start_span', 'end_span']].head())
display(df_train.label.value_counts().rename_axis('categoria').reset_index(name='anotaciones_train'))
overview.to_csv(WORK / 'data_overview.csv', index=False)
print('DATOS LISTOS. Variables: df_train, df_test, df_docs, texts, official_train_ids y official_test_ids.')
print('Esta etapa no ha cargado BERT ni entrenado modelos.')

,recurso,filas,documentos
0,Anotaciones train,33757,750
1,Anotaciones test,11239,250
2,Textos completos,1000,1000


,doc_id,text,label,start_span,end_span
0,S0004-06142005000500011-1,alergias medicamentosas,DISEASE,50,73
1,S0004-06142005000500011-1,fracturas vertebrales y costales,DISEASE,158,190
2,S0004-06142005000500011-1,intervenido de enfermedad de Dupuytren en mano...,PROCEDURE,192,278
3,S0004-06142005000500011-1,enfermedad de Dupuytren,DISEASE,207,230
4,S0004-06142005000500011-1,Diabetes Mellitus tipo II,DISEASE,280,305


,categoria,anotaciones_train
0,PROCEDURE,11065
1,SYMPTOM,9091
2,DISEASE,8065
3,CHEMICAL,3283
4,PROTEIN,2253


DATOS LISTOS. Variables: df_train, df_test, df_docs, texts, official_train_ids y official_test_ids.
Esta etapa no ha cargado BERT ni entrenado modelos.


### Inspección BIO — Distribución de etiquetas por token

Contamos las 11 etiquetas BIO sobre los **750 documentos oficiales de entrenamiento**, sin inspeccionar test. Este alcance incluye los documentos que posteriormente separaremos para validación: sus conteos no deben utilizarse para calcular pesos de la pérdida ni construir el vocabulario del entrenamiento definitivo.

Usamos el tokenizador `es_core_news_sm` de los talleres anteriores. **Estos son tokens de spaCy, no subtokens de BERT**; la alineación con BETO se realizará después. Todavía no hay padding ni tokens especiales.

Aplicamos la misma política BIO plana: alineación estricta por caracteres, entidades más largas primero (desempate por inicio y categoría), eliminación de duplicados y exclusión de solapamientos. Los tokens cubiertos únicamente por anotaciones descartadas se marcan `IGNORE=-100`, en lugar de contarlos como `O`. Cada `B` inicia una entidad retenida y cada `I` la continúa.

La primera tabla presenta el número de tokens por etiqueta y su porcentaje sobre los tokens evaluables (sin `IGNORE`). La segunda muestra la cobertura y las exclusiones. Los conteos se calculan con los datos cargados y se guardan como CSV para revisarlos.

In [5]:
from collections import Counter
import spacy

BIO_TAGS = ['O'] + [f'{prefix}-{label}' for label in LABELS for prefix in ('B', 'I')]
BIO_IGNORE = -100
bio_nlp = spacy.load('es_core_news_sm')
bio_nlp.disable_pipes(*bio_nlp.pipe_names)  # Solo tokenización; no necesitamos POS para contar BIO.


def inspect_bio(document_ids, annotations, document_texts, tokenizer):
    groups = {key: frame for key, frame in annotations.groupby('doc_id')}
    counts = Counter({tag: 0 for tag in BIO_TAGS})
    stats = Counter(documents=0, annotations=0, duplicates=0, unaligned=0,
                    overlap_excluded=0, retained=0, tokens=0, ignored_tokens=0)
    retained_by_label = Counter()
    for doc_id, doc in zip(document_ids, tokenizer.pipe((document_texts[k] for k in document_ids), batch_size=16)):
        gold = ['O'] * len(doc)
        candidates, ignored_spans, seen = [], [], set()
        stats['documents'] += 1
        for row in groups.get(doc_id, annotations.iloc[:0]).itertuples(index=False):
            a, b, label = int(row.start_span), int(row.end_span), row.label
            assert label in LABELS and 0 <= a < b <= len(doc.text)
            stats['annotations'] += 1
            identity = (a, b, label)
            if identity in seen:
                stats['duplicates'] += 1
                continue
            seen.add(identity)
            span = doc.char_span(a, b, alignment_mode='strict')
            if span is None:
                stats['unaligned'] += 1
                ignored_spans.append((a, b))
            else:
                candidates.append((a, b, label, span.start, span.end))
        for a, b, label, start, end in sorted(candidates, key=lambda x: (-(x[1]-x[0]), x[0], x[2])):
            if any(gold[i] != 'O' for i in range(start, end)):
                stats['overlap_excluded'] += 1
                ignored_spans.append((a, b))
                continue
            gold[start] = f'B-{label}'
            gold[start+1:end] = [f'I-{label}'] * (end-start-1)
            stats['retained'] += 1
            retained_by_label[label] += 1
        for token in doc:
            if gold[token.i] == 'O' and any(token.idx < b and token.idx + len(token) > a for a, b in ignored_spans):
                gold[token.i] = BIO_IGNORE
        stats['tokens'] += len(gold)
        stats['ignored_tokens'] += gold.count(BIO_IGNORE)
        counts.update(tag for tag in gold if tag != BIO_IGNORE)
    assert sum(counts.values()) + stats['ignored_tokens'] == stats['tokens']
    assert stats['annotations'] == sum(stats[k] for k in ['duplicates', 'unaligned', 'overlap_excluded', 'retained'])
    assert sum(counts[f'B-{label}'] for label in LABELS) == stats['retained']
    assert all(counts[f'B-{label}'] == retained_by_label[label] for label in LABELS)
    return counts, stats


bio_counts, bio_stats = inspect_bio(official_train_ids, df_train, texts, bio_nlp)
bio_valid_tokens = sum(bio_counts.values())
assert bio_valid_tokens > 0, 'No hay tokens evaluables.'
bio_table = pd.DataFrame([{'etiqueta': tag, 'tokens': bio_counts[tag],
                          'porcentaje_evaluable': 100 * bio_counts[tag] / bio_valid_tokens}
                         for tag in BIO_TAGS])
display(Markdown('#### Distribución BIO: entrenamiento oficial (750 documentos)'))
show_markdown_table(['Etiqueta', 'Tokens', '% de tokens evaluables'],
    [[row.etiqueta, row.tokens, f'{row.porcentaje_evaluable:.2f} %'] for row in bio_table.itertuples(index=False)])
show_markdown_table(['Cobertura / auditoría', 'Cantidad'], [
    ['Documentos inspeccionados', bio_stats['documents']],
    ['Tokens totales (incluye IGNORE)', bio_stats['tokens']],
    ['Tokens evaluables', bio_valid_tokens],
    ['Tokens excluidos: IGNORE (-100)', bio_stats['ignored_tokens']],
    ['Anotaciones originales', bio_stats['annotations']],
    ['Entidades retenidas (número de B)', bio_stats['retained']],
    ['Anotaciones duplicadas', bio_stats['duplicates']],
    ['Anotaciones no alineadas', bio_stats['unaligned']],
    ['Anotaciones excluidas por solapamiento', bio_stats['overlap_excluded']],
])
bio_table.to_csv(WORK / 'bio_distribution_official_train.csv', index=False)
pd.DataFrame([dict(bio_stats)]).to_csv(WORK / 'bio_audit_official_train.csv', index=False)
display(Markdown(f"**Proporción de O:** {100 * bio_counts['O'] / bio_valid_tokens:.2f} % de los tokens evaluables. "
                 'Esta proporción ayuda a interpretar el desbalance;'))
print('Inspección BIO completada. CSV guardados en:', WORK)

#### Distribución BIO: entrenamiento oficial (750 documentos)

| Etiqueta | Tokens | % de tokens evaluables |
| --- | --- | --- |
| O | 211622 | 69.62 % |
| B-CHEMICAL | 2444 | 0.80 % |
| I-CHEMICAL | 329 | 0.11 % |
| B-DISEASE | 7257 | 2.39 % |
| I-DISEASE | 16308 | 5.37 % |
| B-PROCEDURE | 9159 | 3.01 % |
| I-PROCEDURE | 20879 | 6.87 % |
| B-PROTEIN | 1395 | 0.46 % |
| I-PROTEIN | 439 | 0.14 % |
| B-SYMPTOM | 8007 | 2.63 % |
| I-SYMPTOM | 26107 | 8.59 % |

| Cobertura / auditoría | Cantidad |
| --- | --- |
| Documentos inspeccionados | 750 |
| Tokens totales (incluye IGNORE) | 305166 |
| Tokens evaluables | 303946 |
| Tokens excluidos: IGNORE (-100) | 1220 |
| Anotaciones originales | 33757 |
| Entidades retenidas (número de B) | 28262 |
| Anotaciones duplicadas | 0 |
| Anotaciones no alineadas | 254 |
| Anotaciones excluidas por solapamiento | 5241 |

**Proporción de O:** 69.62 % de los tokens evaluables. Esta proporción ayuda a interpretar el desbalance;

Inspección BIO completada. CSV guardados en: /kaggle/working/spaccc_bert


 El análisis de los 750 documentos oficiales de entrenamiento identificó 305,166 tokens, de los cuales 303,946 son evaluables y 1,220 se excluyen mediante
  IGNORE. La etiqueta O representa el 69.62 % de los tokens evaluables, mientras que las etiquetas de entidades (B e I) corresponden al 30.38 %. Este
  predominio es esperable: gran parte del texto aporta contexto sin pertenecer a las categorías clínicas anotadas.

  La distribución evidencia desbalance entre categorías y diferencias en la longitud de las entidades. Se retuvieron 9,159 procedimientos, frente a 1,395
  proteínas; además, la elevada cantidad de etiquetas I-SYMPTOM muestra la presencia de síntomas compuestos por varias palabras.

  De las 33,757 anotaciones originales, se conservaron 28,262 entidades —83.72 %—. Se excluyeron 5,241 anotaciones por solapamiento y 254 por falta de
  alineación. Por tanto, el esquema BIO plano representa una parte del corpus y no captura todas las entidades anidadas o solapadas.

  Estos resultados justifican evaluar principalmente precisión, recall y F1 de entidades completas, junto con métricas por categoría. La exactitud por token
  puede resultar engañosa: predecir siempre O alcanzaría un 69.62 % de exactitud sin identificar ninguna entidad. Los tokens O deben conservarse porque
  aportan contexto;
 


### PASO 04 — Partición fija por documento

Separamos el 20 % de los 750 documentos oficiales de entrenamiento para validación, con `split_seed=42` y el mismo algoritmo de los talleres 1 y 2. Conservamos 600 documentos para entrenamiento, 150 para validación y los 250 documentos oficiales de test para la evaluación final.

Verificamos que las particiones no compartan identificadores ni textos duplicados y que cubran exactamente los documentos oficiales. Comparamos además la huella SHA-256 de las listas completas con la del taller 1; una discrepancia detiene la ejecución.

La partición se hace por documento, no por anotación: todas las entidades y futuras ventanas de un documento permanecerán juntas. Las tablas de anotaciones de cada conjunto se conservan por separado. Sus conteos son anotaciones originales, antes de las exclusiones BIO. El análisis BIO anterior continúa siendo descriptivo sobre los 750 documentos; los futuros pesos de clase deberán calcularse únicamente sobre entrenamiento.

Guardamos los identificadores y la huella en `split.json`, y una tabla de resumen en `split_overview.csv`. Todavía no tokenizamos con BERT ni entrenamos.

In [6]:
# PASO 04 — Misma partición de los talleres anteriores.
import numpy as np

SPLIT_SEED = 42
VALIDATION_FRACTION = 0.20
EXPECTED_SPLIT_SHA256 = 'ab06b4e8dd54e120c2fed93f276f1944a6e67c5227281778c38add2eea4dc497'

assert len(official_train_ids) == 750 and len(official_test_ids) == 250
split_rng = np.random.default_rng(SPLIT_SEED)
ordered_ids = split_rng.permutation(sorted(official_train_ids)).tolist()
n_validation = round(len(ordered_ids) * VALIDATION_FRACTION)
val_ids = sorted(ordered_ids[:n_validation])
train_ids = sorted(ordered_ids[n_validation:])
test_ids = sorted(official_test_ids)

split_ids = {'train': train_ids, 'validation': val_ids, 'test': test_ids}
assert (len(train_ids), len(val_ids), len(test_ids)) == (600, 150, 250)
assert set(train_ids) | set(val_ids) == set(official_train_ids)
assert all(len(ids) == len(set(ids)) for ids in split_ids.values())
for left, right in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    assert not set(split_ids[left]) & set(split_ids[right]), f'Documentos compartidos: {left}/{right}'
    left_texts = {fingerprint(texts[k]) for k in split_ids[left]}
    right_texts = {fingerprint(texts[k]) for k in split_ids[right]}
    assert not left_texts & right_texts, f'Textos duplicados: {left}/{right}'

# Usar las mismas claves y serialización que el taller 1 permite comparar la huella.
split_payload = {'train': train_ids, 'validation': val_ids, 'official_test': test_ids}
split_sha256 = hashlib.sha256(json.dumps(split_payload, sort_keys=True).encode()).hexdigest()
assert split_sha256 == EXPECTED_SPLIT_SHA256, 'La partición no coincide con los talleres anteriores.'

# df_train conserva las anotaciones oficiales de los 750 documentos.
# Estos nuevos objetos evitan sobrescribir los datos usados en la inspección anterior.
annotations_by_split = {
    'train': df_train[df_train.doc_id.isin(train_ids)].copy(),
    'validation': df_train[df_train.doc_id.isin(val_ids)].copy(),
    'test': df_test[df_test.doc_id.isin(test_ids)].copy(),
}
documents_by_split = {name: df_docs[df_docs.doc_id.isin(ids)].copy()
                      for name, ids in split_ids.items()}
assert len(annotations_by_split['train']) + len(annotations_by_split['validation']) == len(df_train)
assert len(annotations_by_split['test']) == len(df_test)
assert all(set(documents_by_split[name].doc_id) == set(ids) for name, ids in split_ids.items())

split_overview = pd.DataFrame([
    {'conjunto': name, 'documentos': len(ids), 'anotaciones_originales': len(annotations_by_split[name])}
    for name, ids in split_ids.items()
])
REPORT['split'] = {'split_seed': SPLIT_SEED, 'validation_fraction': VALIDATION_FRACTION,
    'train_documents': len(train_ids), 'validation_documents': len(val_ids),
    'test_documents': len(test_ids), 'sha256': split_sha256,
    'matches_previous_workshops': True, 'test_used_for_training': False}
REPORT['versions']['numpy'] = np.__version__
REPORT['status'] = 'DATOS_Y_PARTICION_VERIFICADOS'
(WORK / 'split.json').write_text(json.dumps({**split_payload, **REPORT['split']}, indent=2), encoding='utf-8')
split_overview.to_csv(WORK / 'split_overview.csv', index=False)
(WORK / 'data_summary.json').write_text(json.dumps(REPORT, indent=2, ensure_ascii=False), encoding='utf-8')

display(Markdown('### PASO 04 OK — Partición verificada'))
show_markdown_table(['Conjunto', 'Documentos', 'Anotaciones originales'],
                    split_overview.itertuples(index=False, name=None))

print('Partición guardada en:', WORK / 'split.json')
print('Disponibles: train_ids, val_ids, test_ids, annotations_by_split y documents_by_split.')

### PASO 04 OK — Partición verificada

| Conjunto | Documentos | Anotaciones originales |
| --- | --- | --- |
| train | 600 | 26849 |
| validation | 150 | 6908 |
| test | 250 | 11239 |

Partición guardada en: /kaggle/working/spaccc_bert/split.json
Disponibles: train_ids, val_ids, test_ids, annotations_by_split y documents_by_split.


### PASO 05 — Tokenizador de BETO y alineación exploratoria

Cargamos **solo el tokenizador y la configuración** de `dccuchile/bert-base-spanish-wwm-cased`; no descargamos pesos ni entrenamos BERT. La configuración permite conocer el límite posicional del modelo, incluso si el tokenizador publica un límite indefinido.

Usamos el tokenizador rápido de Hugging Face y `is_split_into_words=True` para conservar el vínculo con los tokens de spaCy. `word_ids()` identifica el token de origen de cada subtoken. La política de supervisión será: **etiquetar solo el primer subtoken de cada token de spaCy**; continuaciones, tokens especiales y posiciones excluidas reciben `-100`. Esto evita dar más peso a una palabra solo porque se fragmenta más.

Analizamos entrenamiento y validación completos sin truncamiento y sin padding. Los documentos largos todavía no se envían al modelo. Medimos fragmentación, posiciones sin supervisión y tokens que BETO omite al normalizar (por ejemplo, espacios). Estos últimos deben auditarse antes de comparar las métricas con los talleres anteriores.

Las tablas se calculan con los datos reales. Referencias: [tokenizadores de Hugging Face](https://huggingface.co/docs/transformers/main_classes/tokenizer) y [alineación para clasificación por token](https://huggingface.co/docs/transformers/tasks/token_classification).

In [7]:
# Ejecutar una vez por sesión; 
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.46,<5'])

import transformers
from transformers import AutoTokenizer, AutoConfig

BETO_CACHE = WORK / 'hf_cache'
TOKENIZER_DIR = WORK / 'beto_tokenizer'
# Tras la primera descarga reutilizamos una copia local con la revisión registrada.
if (TOKENIZER_DIR / 'tokenizer_config.json').exists() and (TOKENIZER_DIR / 'config.json').exists():
    beto_config = AutoConfig.from_pretrained(TOKENIZER_DIR, local_files_only=True)
    beto_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, use_fast=True, local_files_only=True)
else:
    beto_config = AutoConfig.from_pretrained(MODEL_CHECKPOINT, cache_dir=BETO_CACHE)
    beto_revision = getattr(beto_config, '_commit_hash', None)
    beto_tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True,
        revision=beto_revision or 'main', cache_dir=BETO_CACHE)
    TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
    beto_tokenizer.save_pretrained(TOKENIZER_DIR)
    beto_config.save_pretrained(TOKENIZER_DIR)
    (TOKENIZER_DIR / 'source.json').write_text(json.dumps({'checkpoint': MODEL_CHECKPOINT,
        'revision': beto_revision}, indent=2), encoding='utf-8')
assert beto_tokenizer.is_fast, 'Se necesita un tokenizador rápido para utilizar word_ids().'
assert beto_tokenizer.pad_token_id is not None
beto_position_limit = int(beto_config.max_position_embeddings)
beto_tokenizer_limit = int(beto_tokenizer.model_max_length)
BETO_MAX_LENGTH = min(beto_position_limit, beto_tokenizer_limit)
BETO_SPECIAL_TOKENS = beto_tokenizer.num_special_tokens_to_add(pair=False)
BETO_CONTENT_CAPACITY = BETO_MAX_LENGTH - BETO_SPECIAL_TOKENS
assert BETO_CONTENT_CAPACITY > 0
show_markdown_table(['Propiedad', 'Valor'], [
    ['Checkpoint', MODEL_CHECKPOINT], ['Tokenizador rápido', beto_tokenizer.is_fast],
    ['Vocabulario (incluye tokens añadidos)', len(beto_tokenizer)],
    ['Límite de entrada efectivo', BETO_MAX_LENGTH],
    ['Tokens especiales por secuencia', BETO_SPECIAL_TOKENS],
    ['Capacidad de contenido por secuencia', BETO_CONTENT_CAPACITY],
    ['Política BIO', 'Solo primer subtoken; resto: IGNORE (-100)'],
])

| Propiedad | Valor |
| --- | --- |
| Checkpoint | dccuchile/bert-base-spanish-wwm-cased |
| Tokenizador rápido | True |
| Vocabulario (incluye tokens añadidos) | 31002 |
| Límite de entrada efectivo | 512 |
| Tokens especiales por secuencia | 2 |
| Capacidad de contenido por secuencia | 510 |
| Política BIO | Solo primer subtoken; resto: IGNORE (-100) |

#### PASO 05A — Preparar etiquetas de referencia y analizar longitudes

Reutilizamos la función de inspección BIO, ahora conservando tokens, etiquetas y posiciones por documento en `bio_examples_by_split`. Las listas de identificadores son las del PASO 04. No utilizamos `test`.

`beto_analysis_encodings` contiene documentos completos para inspección, incluidos los que exceden el límite: **no son todavía lotes válidos para BERT**. La comparación de longitudes incluye `[CLS]` y `[SEP]`, sin padding. Los porcentajes de etiquetas BERT usan solamente posiciones supervisadas; las posiciones `-100` se cuentan aparte.

In [8]:
# Compatibilidad con sesiones que ejecutaron la versión anterior de la inspección BIO.
# Esta copia usa la misma política que la celda de inspección y solo se activa si hace falta.
import inspect
if 'inspect_bio' not in globals() or 'return_examples' not in inspect.signature(inspect_bio).parameters:
    print('Actualizando inspect_bio para devolver tokens, offsets y etiquetas por documento.')
    def inspect_bio(document_ids, annotations, document_texts, tokenizer, return_examples=False):
        groups = {key: frame for key, frame in annotations.groupby('doc_id')}
        counts = Counter({tag: 0 for tag in BIO_TAGS})
        stats = Counter(documents=0, annotations=0, duplicates=0, unaligned=0,
                        overlap_excluded=0, retained=0, tokens=0, ignored_tokens=0)
        retained_by_label = Counter()
        examples = []
        for doc_id, doc in zip(document_ids, tokenizer.pipe((document_texts[k] for k in document_ids), batch_size=16)):
            gold = ['O'] * len(doc)
            candidates, ignored_spans, seen = [], [], set()
            stats['documents'] += 1
            for row in groups.get(doc_id, annotations.iloc[:0]).itertuples(index=False):
                a, b, label = int(row.start_span), int(row.end_span), row.label
                assert label in LABELS and 0 <= a < b <= len(doc.text)
                stats['annotations'] += 1
                identity = (a, b, label)
                if identity in seen:
                    stats['duplicates'] += 1
                    continue
                seen.add(identity)
                span = doc.char_span(a, b, alignment_mode='strict')
                if span is None:
                    stats['unaligned'] += 1
                    ignored_spans.append((a, b))
                else:
                    candidates.append((a, b, label, span.start, span.end))
            for a, b, label, start, end in sorted(candidates, key=lambda x: (-(x[1]-x[0]), x[0], x[2])):
                if any(gold[i] != 'O' for i in range(start, end)):
                    stats['overlap_excluded'] += 1
                    ignored_spans.append((a, b))
                    continue
                gold[start] = f'B-{label}'
                gold[start+1:end] = [f'I-{label}'] * (end-start-1)
                stats['retained'] += 1
                retained_by_label[label] += 1
            for token in doc:
                if gold[token.i] == 'O' and any(token.idx < b and token.idx + len(token) > a for a, b in ignored_spans):
                    gold[token.i] = BIO_IGNORE
            if return_examples:
                examples.append({'doc_id': doc_id, 'tokens': [token.text for token in doc],
                                 'offsets': [(token.idx, token.idx + len(token)) for token in doc],
                                 'tags': gold})
            stats['tokens'] += len(gold)
            stats['ignored_tokens'] += gold.count(BIO_IGNORE)
            counts.update(tag for tag in gold if tag != BIO_IGNORE)
        assert sum(counts.values()) + stats['ignored_tokens'] == stats['tokens']
        assert stats['annotations'] == sum(stats[k] for k in ['duplicates', 'unaligned', 'overlap_excluded', 'retained'])
        assert sum(counts[f'B-{label}'] for label in LABELS) == stats['retained']
        assert all(counts[f'B-{label}'] == retained_by_label[label] for label in LABELS)
        return (counts, stats, examples) if return_examples else (counts, stats)

assert REPORT['split']['sha256'] == EXPECTED_SPLIT_SHA256, 'Ejecuta primero el PASO 04.'
BIO_LABEL2ID = {tag: i for i, tag in enumerate(BIO_TAGS)}
BIO_ID2LABEL = {i: tag for tag, i in BIO_LABEL2ID.items()}


def align_first_subtoken(encoding, tags):
    labels, seen = [], set()
    for word_id in encoding.word_ids():
        if word_id is None or word_id in seen:
            labels.append(BIO_IGNORE)
        else:
            seen.add(word_id)
            tag = tags[word_id]
            labels.append(BIO_IGNORE if tag == BIO_IGNORE else BIO_LABEL2ID[tag])
    assert len(labels) == len(encoding['input_ids'])
    return labels


bio_examples_by_split, beto_analysis_encodings = {}, {}
beto_rows, beto_label_rows, beto_missing_rows = [], [], []
for split_name in ('train', 'validation'):
    _, _, examples = inspect_bio(split_ids[split_name], annotations_by_split[split_name],
                                 texts, bio_nlp, return_examples=True)
    bio_examples_by_split[split_name] = examples
    beto_analysis_encodings[split_name] = []
    split_label_counts = Counter({tag: 0 for tag in BIO_TAGS})
    for example in examples:
        encoding = beto_tokenizer(example['tokens'], is_split_into_words=True,
            add_special_tokens=True, truncation=False, padding=False,
            return_special_tokens_mask=True, verbose=False)
        word_ids = encoding.word_ids()
        multiplicities = Counter(w for w in word_ids if w is not None)
        aligned = align_first_subtoken(encoding, example['tags'])
        encoding['labels'] = aligned
        beto_analysis_encodings[split_name].append(encoding)
        split_label_counts.update(BIO_ID2LABEL[y] for y in aligned if y != BIO_IGNORE)
        missing = set(range(len(example['tokens']))) - set(multiplicities)
        for index in sorted(missing):
            beto_missing_rows.append({'conjunto': split_name, 'doc_id': example['doc_id'],
                'word_id': index, 'token': repr(example['tokens'][index]),
                'etiqueta': example['tags'][index], 'es_espacio': example['tokens'][index].isspace()})
        entity_words = [i for i, tag in enumerate(example['tags']) if tag != BIO_IGNORE and tag != 'O']
        total_content = sum(multiplicities.values())
        row = {'conjunto': split_name, 'doc_id': example['doc_id'],
            'tokens_spacy': len(example['tokens']), 'tokens_representados': len(multiplicities),
            'subtokens_contenido': total_content, 'longitud_con_especiales': len(encoding['input_ids']),
            'tokens_fragmentados': sum(v > 1 for v in multiplicities.values()),
            'tokens_entidad_representados': sum(i in multiplicities for i in entity_words),
            'subtokens_entidad': sum(multiplicities[i] for i in entity_words),
            'tokens_sin_subtokens': len(missing),
            'tokens_entidad_sin_subtokens': sum(i in missing for i in entity_words),
            'subtokens_UNK': encoding['input_ids'].count(beto_tokenizer.unk_token_id),
            'posiciones_supervisadas': sum(y != BIO_IGNORE for y in aligned),
            'posiciones_ignoradas': aligned.count(BIO_IGNORE),
            'excede_limite': len(encoding['input_ids']) > BETO_MAX_LENGTH}
        assert row['posiciones_supervisadas'] + row['posiciones_ignoradas'] == row['longitud_con_especiales']
        assert row['posiciones_supervisadas'] == sum(example['tags'][i] != BIO_IGNORE for i in multiplicities)
        beto_rows.append(row)
    supervised_total = sum(split_label_counts.values())
    beto_label_rows.extend({'conjunto': split_name, 'etiqueta': tag, 'posiciones': split_label_counts[tag],
        'porcentaje_supervisado': 100 * split_label_counts[tag] / supervised_total if supervised_total else 0.0}
        for tag in BIO_TAGS)

beto_lengths = pd.DataFrame(beto_rows)
beto_label_distribution = pd.DataFrame(beto_label_rows)
beto_missing_tokens = pd.DataFrame(beto_missing_rows, columns=['conjunto', 'doc_id', 'word_id', 'token', 'etiqueta', 'es_espacio'])
length_summary, fragmentation_summary = [], []
for split_name, group in beto_lengths.groupby('conjunto', sort=False):
    sizes = group.longitud_con_especiales
    length_summary.append([split_name, len(group), int(sizes.min()), round(float(sizes.median()), 1),
        round(float(sizes.quantile(0.95)), 1), int(sizes.max()), int(group.excede_limite.sum()),
        f'{100 * group.excede_limite.mean():.2f} %'])
    represented = int(group.tokens_representados.sum())
    entities = int(group.tokens_entidad_representados.sum())
    fragmentation_summary.append([split_name,
        f'{group.subtokens_contenido.sum() / max(represented, 1):.2f}',
        f'{100 * group.tokens_fragmentados.sum() / max(represented, 1):.2f} %',
        f'{group.subtokens_entidad.sum() / max(entities, 1):.2f}',
        int(group.tokens_sin_subtokens.sum()), int(group.tokens_entidad_sin_subtokens.sum()),
        int(group.subtokens_UNK.sum())])
display(Markdown('#### Longitudes completas, incluidos los tokens especiales'))
show_markdown_table(['Conjunto', 'Documentos', 'Mínimo', 'Mediana', 'P95', 'Máximo', 'Exceden límite', '% excede'], length_summary)
display(Markdown('#### Fragmentación y cobertura'))
show_markdown_table(['Conjunto', 'Subtokens/token representado', '% tokens fragmentados',
    'Subtokens/token de entidad', 'Tokens omitidos', 'Tokens de entidad omitidos', 'Subtokens UNK'], fragmentation_summary)
display(Markdown('Los tokens omitidos se registran en `beto_missing_tokens.csv`. '
    'No se asignan etiquetas artificiales a posiciones que el tokenizador no produce. '
    'Si hay tokens de entidad omitidos, habrá que resolverlos antes de entrenar.'))
for split_name in ('train', 'validation'):
    display(Markdown(f'#### Etiquetas supervisadas tras tokenizar: {split_name}'))
    show_markdown_table(['Etiqueta', 'Posiciones', '% supervisado'],
        [[r.etiqueta, r.posiciones, f'{r.porcentaje_supervisado:.2f} %']
         for r in beto_label_distribution[beto_label_distribution.conjunto == split_name].itertuples(index=False)])
show_markdown_table(['Conjunto', 'Posiciones supervisadas', 'Posiciones IGNORE (sin padding)'],
    [[name, int(g.posiciones_supervisadas.sum()), int(g.posiciones_ignoradas.sum())]
     for name, g in beto_lengths.groupby('conjunto', sort=False)])

Actualizando inspect_bio para devolver tokens, offsets y etiquetas por documento.


#### Longitudes completas, incluidos los tokens especiales

| Conjunto | Documentos | Mínimo | Mediana | P95 | Máximo | Exceden límite | % excede |
| --- | --- | --- | --- | --- | --- | --- | --- |
| train | 600 | 110 | 511.5 | 1072.1 | 1792 | 297 | 49.50 % |
| validation | 150 | 139 | 530.5 | 1125.2 | 1514 | 81 | 54.00 % |

#### Fragmentación y cobertura

| Conjunto | Subtokens/token representado | % tokens fragmentados | Subtokens/token de entidad | Tokens omitidos | Tokens de entidad omitidos | Subtokens UNK |
| --- | --- | --- | --- | --- | --- | --- |
| train | 1.39 | 21.82 % | 1.83 | 3042 | 0 | 540 |
| validation | 1.39 | 21.78 % | 1.84 | 779 | 0 | 116 |

...

#### Etiquetas supervisadas tras tokenizar: train

| Etiqueta | Posiciones | % supervisado |
| --- | --- | --- |
| O | 164593 | 69.10 % |
| B-CHEMICAL | 1867 | 0.78 % |
| I-CHEMICAL | 251 | 0.11 % |
| B-DISEASE | 5866 | 2.46 % |
| I-DISEASE | 13097 | 5.50 % |
| B-PROCEDURE | 7253 | 3.05 % |
| I-PROCEDURE | 16851 | 7.07 % |
| B-PROTEIN | 1090 | 0.46 % |
| I-PROTEIN | 336 | 0.14 % |
| B-SYMPTOM | 6325 | 2.66 % |
| I-SYMPTOM | 20658 | 8.67 % |

#### Etiquetas supervisadas tras tokenizar: validation

| Etiqueta | Posiciones | % supervisado |
| --- | --- | --- |
| O | 43208 | 69.76 % |
| B-CHEMICAL | 577 | 0.93 % |
| I-CHEMICAL | 78 | 0.13 % |
| B-DISEASE | 1391 | 2.25 % |
| I-DISEASE | 3211 | 5.18 % |
| B-PROCEDURE | 1906 | 3.08 % |
| I-PROCEDURE | 4028 | 6.50 % |
| B-PROTEIN | 305 | 0.49 % |
| I-PROTEIN | 103 | 0.17 % |
| B-SYMPTOM | 1682 | 2.72 % |
| I-SYMPTOM | 5449 | 8.80 % |

| Conjunto | Posiciones supervisadas | Posiciones IGNORE (sin padding) |
| --- | --- | --- |
| train | 238187 | 94759 |
| validation | 61938 | 24532 |

### Conclusiones del análisis de tokenización con BETO

  El análisis del tokenizador BETO mostró que una parte importante de los documentos supera el límite de 512 posiciones: el 49.50 % de los documentos de
  entrenamiento y el 54.00 % de validación. Por esta razón, no es adecuado truncar los documentos directamente; será necesario procesarlos mediante ventanas
  con solapamiento.

  La fragmentación de palabras fue moderada. En ambos conjuntos, aproximadamente el 21.8 % de los tokens se dividió en varios subtokens, con un promedio de
  1.39 subtokens por token. En las entidades clínicas la fragmentación fue mayor, con aproximadamente 1.83 subtokens por token de entidad. A pesar de esto, no
  se omitió ningún token perteneciente a una entidad, lo que confirma una cobertura completa de las menciones anotadas.

  La distribución de etiquetas supervisadas fue consistente entre entrenamiento y validación. La etiqueta O representó el 69.10 % de las posiciones
  supervisadas en entrenamiento y el 69.76 % en validación. Este desbalance es esperado en NER y confirma que la exactitud por token no debe utilizarse como
  métrica principal.

  Las posiciones IGNORE corresponden a tokens especiales, subtokens posteriores al primero y posiciones que no deben contribuir directamente a la pérdida.
  Esta estrategia permite mantener la información contextual de los subtokens sin sobreponderar las palabras fragmentadas.

  En conclusión, BETO ofrece una cobertura adecuada de las entidades clínicas y una distribución estable entre las particiones. El principal reto para la
  siguiente etapa será construir ventanas de hasta 512 posiciones, conservar la alineación BIO y evitar duplicar entidades en las zonas de solapamiento.

#### PASO 05B — Ejemplos clínicos y correspondencia de etiquetas

Mostramos un fragmento de entrenamiento alrededor de una entidad retenida, prefiriendo un token de entidad que se divida en varios subtokens. `word_id` identifica la posición dentro del fragmento de spaCy. La etiqueta original se muestra en todas sus partes para facilitar la lectura, pero **solo el primer subtoken aporta una etiqueta a la pérdida**.

En WordPiece, el prefijo `##` indica una continuación de palabra; no es parte del texto clínico. `IGNORE` en una continuación no significa que el modelo no pueda atender a ella: solamente se excluye su contribución directa a la pérdida.

In [9]:
example_to_show = None
for example, full_encoding in zip(bio_examples_by_split['train'], beto_analysis_encodings['train']):
    counts = Counter(w for w in full_encoding.word_ids() if w is not None)
    candidates = [i for i, tag in enumerate(example['tags'])
                  if tag != BIO_IGNORE and tag != 'O' and counts[i] > 1]
    if candidates:
        example_to_show, focus = example, candidates[0]
        break
assert example_to_show is not None, 'No se encontró una entidad fragmentada para inspeccionar.'
# Incluir la entidad completa y algo de contexto (sin comenzar dentro de una entidad).
left, right = focus, focus + 1
while left > 0 and str(example_to_show['tags'][left]).startswith('I-'):
    left -= 1
while right < len(example_to_show['tags']) and str(example_to_show['tags'][right]).startswith('I-'):
    right += 1
left = max(0, left - 3)
while left > 0 and str(example_to_show['tags'][left]).startswith('I-'):
    left -= 1
right = min(len(example_to_show['tokens']), right + 3)
while right < len(example_to_show['tags']) and str(example_to_show['tags'][right]).startswith('I-'):
    right += 1
sample_words = example_to_show['tokens'][left:right]
sample_tags = example_to_show['tags'][left:right]
sample_encoding = beto_tokenizer(sample_words, is_split_into_words=True, truncation=False)
sample_labels = align_first_subtoken(sample_encoding, sample_tags)
sample_rows = []
for subtoken, word_id, label_id in zip(sample_encoding.tokens(), sample_encoding.word_ids(), sample_labels):
    sample_rows.append([subtoken, '—' if word_id is None else word_id,
        'Especial' if word_id is None else sample_words[word_id],
        '—' if word_id is None else sample_tags[word_id],
        'IGNORE (-100)' if label_id == BIO_IGNORE else BIO_ID2LABEL[label_id]])
display(Markdown(f"**Documento de entrenamiento:** `{example_to_show['doc_id']}`"))
show_markdown_table(['Subtoken BETO', 'word_id local', 'Token spaCy', 'BIO original', 'Etiqueta supervisada'], sample_rows)

**Documento de entrenamiento:** `S0004-06142005000700014-1`

| Subtoken BETO | word_id local | Token spaCy | BIO original | Etiqueta supervisada |
| --- | --- | --- | --- | --- |
| [CLS] | — | Especial | — | IGNORE (-100) |
| activa | 0 | activa | O | O |
| que | 1 | que | O | O |
| refiere | 2 | refiere | O | O |
| dolores | 3 | dolores | B-SYMPTOM | B-SYMPTOM |
| os | 4 | osteoarticulares | I-SYMPTOM | I-SYMPTOM |
| ##teo | 4 | osteoarticulares | I-SYMPTOM | IGNORE (-100) |
| ##art | 4 | osteoarticulares | I-SYMPTOM | IGNORE (-100) |
| ##icular | 4 | osteoarticulares | I-SYMPTOM | IGNORE (-100) |
| ##es | 4 | osteoarticulares | I-SYMPTOM | IGNORE (-100) |
| de | 5 | de | O | O |
| localización | 6 | localización | O | O |
| variable | 7 | variable | O | O |
| [SEP] | — | Especial | — | IGNORE (-100) |

### Interpretación del PASO 5B

  El ejemplo muestra cómo BETO transforma la entidad clínica «dolores osteoarticulares» en varios subtokens. La palabra «dolores» conserva la etiqueta B-
  SYMPTOM, mientras que «osteoarticulares» se divide en os, ##teo, ##art, ##icular y ##es.

  Solo el primer subtoken recibe I-SYMPTOM; los subtokens restantes se marcan como IGNORE (-100). Esta decisión es adecuada porque las anotaciones originales
  están definidas a nivel de palabras completas, mientras que BETO utiliza subtokens. De esta forma, BERT procesa todos los subtokens y aprovecha su contexto,
  pero cada palabra contribuye una sola vez a la función de pérdida, evitando que las palabras fragmentadas tengan un peso desproporcionado.

  Los tokens especiales [CLS] y [SEP] también se ignoran durante el entrenamiento. En la evaluación, la etiqueta de la palabra se recupera a partir de su
  primer subtoken. Esta política conserva la correspondencia con las etiquetas BIO originales y evita modificar artificialmente la distribución de las clases.

#### PASO 05C — Decisión de ventanas para documentos largos

Los resultados del PASO 5A muestran que el truncamiento simple perdería demasiado contexto: el 49.50 % de los documentos de entrenamiento y el 54.00 % de validación superan las 512 posiciones. Por ello, 512 será la longitud total de cada ventana, incluidos `[CLS]` y `[SEP]`; la capacidad de contenido es `512 - BETO_SPECIAL_TOKENS`.

Los diagnósticos de 128 y 256 posiciones se conservan únicamente como referencia. La configuración propuesta para el PASO 06 es una ventana total de 512 posiciones con un solapamiento de 64 subtokens. El solapamiento no es una afirmación de que 64 sea óptimo: permite probar la construcción y medir cobertura antes de ajustar ese valor.

Al construir las ventanas respetaremos los límites de los tokens originales de spaCy, conservaremos identificadores y offsets, y comprobaremos que todos los tokens de entidades estén cubiertos. Si una entidad aparece en dos ventanas, la evaluación deberá deduplicarla mediante sus offsets y categoría. **No utilizaremos las anotaciones gold para elegir los cortes de evaluación.** La cantidad de subtokens que se perdería al truncar es un diagnóstico de cobertura; no equivale al número de entidades perdidas. La construcción efectiva de ventanas será el PASO 06.

In [10]:
BETO_WINDOW_LENGTH = 512
assert BETO_WINDOW_LENGTH <= BETO_MAX_LENGTH, 'BETO no admite ventanas de 512 posiciones.'
BETO_OVERLAP_SUBTOKENS = 64
assert BETO_SPECIAL_TOKENS < BETO_WINDOW_LENGTH <= BETO_MAX_LENGTH
assert 0 <= BETO_OVERLAP_SUBTOKENS < BETO_WINDOW_LENGTH - BETO_SPECIAL_TOKENS
window_rows = []
for limit in sorted(set([v for v in (128, 256, 512) if v <= BETO_MAX_LENGTH] + [BETO_WINDOW_LENGTH])):
    capacity = limit - BETO_SPECIAL_TOKENS
    for split_name, group in beto_lengths.groupby('conjunto', sort=False):
        lost = (group.subtokens_contenido - capacity).clip(lower=0)
        window_rows.append({'conjunto': split_name, 'longitud_total': limit,
            'capacidad_contenido': capacity,
            'documentos_que_requieren_ventanas': int((group.longitud_con_especiales > limit).sum()),
            'subtokens_perdidos_si_truncamos': int(lost.sum()),
            'porcentaje_contenido_perdido': 100 * float(lost.sum()) / max(int(group.subtokens_contenido.sum()), 1)})
beto_window_diagnostics = pd.DataFrame(window_rows)
show_markdown_table(['Conjunto', 'Longitud total', 'Capacidad contenido', 'Documentos largos',
    'Subtokens perdidos al truncar', '% contenido perdido'],
    [[r.conjunto, r.longitud_total, r.capacidad_contenido, r.documentos_que_requieren_ventanas,
      r.subtokens_perdidos_si_truncamos, f'{r.porcentaje_contenido_perdido:.2f} %']
     for r in beto_window_diagnostics.itertuples(index=False)])
for filename, table in [('beto_document_lengths', beto_lengths),
                        ('beto_supervised_labels', beto_label_distribution),
                        ('beto_missing_tokens', beto_missing_tokens),
                        ('beto_window_diagnostics', beto_window_diagnostics)]:
    table.to_csv(WORK / f'{filename}.csv', index=False)
REPORT['versions']['transformers'] = transformers.__version__
REPORT['tokenization'] = {'checkpoint': MODEL_CHECKPOINT,
    'source': json.loads((TOKENIZER_DIR / 'source.json').read_text()),
    'tokenizer_files_sha256': {f.name: hashlib.sha256(f.read_bytes()).hexdigest()
                               for f in TOKENIZER_DIR.iterdir() if f.is_file()},
    'max_length': BETO_MAX_LENGTH, 'special_tokens': BETO_SPECIAL_TOKENS,
    'alignment_policy': 'first-subtoken-only; specials and excluded positions ignored',
    'analyzed_splits': ['train', 'validation'], 'test_analyzed': False,
    'truncation': False, 'padding': False, 'model_weights_loaded': False,
    'window_length_proposed': BETO_WINDOW_LENGTH, 'overlap_subtokens_proposed': BETO_OVERLAP_SUBTOKENS,
    'window_decision': '512 posiciones totales; 64 subtokens de solapamiento; diagnósticos 128/256 solo informativos',
    'missing_entity_tokens': int(beto_lengths.tokens_entidad_sin_subtokens.sum()),
    'window_diagnostics': window_rows}
REPORT['status'] = 'TOKENIZADOR_ANALIZADO_VENTANAS_PENDIENTES'
(WORK / 'data_summary.json').write_text(json.dumps(REPORT, indent=2, ensure_ascii=False), encoding='utf-8')
display(Markdown('**PASO 05 OK.** Tokenizador y análisis guardados. '
    'Los documentos completos aún deben convertirse en ventanas antes de enviarlos a BERT.'))

| Conjunto | Longitud total | Capacidad contenido | Documentos largos | Subtokens perdidos al truncar | % contenido perdido |
| --- | --- | --- | --- | --- | --- |
| train | 128 | 126 | 599 | 256164 | 77.22 % |
| validation | 128 | 126 | 150 | 67270 | 78.07 % |
| train | 256 | 254 | 547 | 181864 | 54.82 % |
| validation | 256 | 254 | 138 | 48925 | 56.78 % |
| train | 512 | 510 | 297 | 72379 | 21.82 % |
| validation | 512 | 510 | 81 | 20917 | 24.27 % |

**PASO 05 OK.** Tokenizador y análisis guardados. Los documentos completos aún deben convertirse en ventanas antes de enviarlos a BERT.

### Interpretación del análisis de ventanas

  Los resultados muestran que una gran proporción de los documentos clínicos supera la longitud máxima de 512 posiciones de BETO. En particular, el 49.5 % de
  los documentos de entrenamiento y el 54 % de los documentos de validación requieren más de una ventana.

  Una truncación directa a 128 o 256 posiciones perdería entre el 54.82 % y el 78.07 % del contenido, por lo que estas longitudes no son adecuadas. Incluso
  con 512 posiciones se perdería aproximadamente el 21.82 % del contenido de entrenamiento y el 24.27 % de validación si solo se conservara el primer
  segmento.

  Estos resultados justifican procesar los documentos mediante ventanas de 512 posiciones con un solapamiento de 64 subtokens. Esta estrategia permitirá
  conservar una mayor cantidad de contexto y reducir el riesgo de cortar entidades clínicas. La cantidad de subtokens perdidos representa contenido
  potencialmente descartado y no equivale directamente al número de entidades perdidas; por ello, el siguiente paso debe verificar la cobertura de las
  anotaciones y eliminar duplicados en las zonas de solapamiento.


### PASO 06 — Construcción y validación de ventanas BERT

Convertimos cada codificación completa de entrenamiento y validación en ventanas de **512 posiciones totales**: hasta 510 subtokens de contenido más `[CLS]` y `[SEP]`. Las ventanas consecutivas comparten aproximadamente 64 subtokens; el inicio se ajusta al límite de un token original de spaCy para no partir una palabra.

Cada ventana conserva `input_ids`, `attention_mask`, `token_type_ids`, `word_ids`, `labels`, offsets y un identificador de documento/ventana. Las etiquetas usan la política del PASO 5B: primer subtoken supervisado; continuaciones, tokens especiales y posiciones excluidas en `IGNORE=-100`. Una entidad puede aparecer en más de una ventana, así que la evaluación deberá deduplicarla mediante documento, offsets y categoría.

No usamos etiquetas gold para decidir cortes. Antes de continuar comprobamos que ninguna ventana exceda el límite y que todos los tokens spaCy pertenecientes a entidades estén cubiertos; BETO puede omitir tokens de espacio. Test permanece reservado.

In [11]:
# PASO 06 — Ventanas con límites de tokens spaCy y solapamiento de 64 subtokens.
assert BETO_WINDOW_LENGTH == 512 and BETO_OVERLAP_SUBTOKENS == 64

def build_windows(example, encoding, split_name, document_index):
    full_word_ids = encoding.word_ids()
    positions = [i for i, wid in enumerate(full_word_ids) if wid is not None]
    groups = {}
    for pos in positions: groups.setdefault(full_word_ids[pos], []).append(pos)
    words = sorted(groups)
    represented = set(words)
    missing_words = set(range(len(example['tokens']))) - represented
    # BETO omite espacios que spaCy conserva; una omisión de entidad sí es un error.
    missing_entities = [wid for wid in missing_words if example['tags'][wid] not in (BIO_IGNORE, 'O')]
    assert not missing_entities, f'Token de entidad sin representación: {example["doc_id"]}'
    capacity = BETO_WINDOW_LENGTH - BETO_SPECIAL_TOKENS
    result, cursor, index, covered = [], 0, 0, set()
    while cursor < len(words):
        end = cursor; count = 0
        while end < len(words):
            new_count = count + len(groups[words[end]])
            if new_count > capacity and end > cursor: break
            if new_count > capacity: raise ValueError(f'Palabra mayor que la ventana: {example["doc_id"]}')
            count, end = new_count, end + 1
        selected = words[cursor:end]
        selected_positions = [p for wid in selected for p in groups[wid]]
        ids = [beto_tokenizer.cls_token_id] + [encoding['input_ids'][p] for p in selected_positions] + [beto_tokenizer.sep_token_id]
        labels = [BIO_IGNORE] + [encoding['labels'][p] for p in selected_positions] + [BIO_IGNORE]
        covered.update(selected)
        result.append({'window_id': f'{split_name}-{document_index}-{index}', 'doc_id': example['doc_id'],
            'split': split_name, 'window_index': index, 'start_word': selected[0], 'end_word': selected[-1],
            'input_ids': ids, 'attention_mask': [1]*len(ids),
            'token_type_ids': [encoding['token_type_ids'][0] if 'token_type_ids' in encoding else 0]*len(ids),
            'word_ids': [None] + [wid for wid in selected for _ in groups[wid]] + [None], 'labels': labels,
            'offsets': [example['offsets'][wid] for wid in selected],
            'tokens': [example['tokens'][wid] for wid in selected],
            'entity_word_ids': [wid for wid in selected if example['tags'][wid] not in (BIO_IGNORE, 'O')]})
        index += 1
        if end == len(words): break
        target = max(groups[selected[0]][0] + 1, selected_positions[-1] + 1 - BETO_OVERLAP_SUBTOKENS)
        next_cursor = next((j for j in range(end, len(words)) if groups[words[j]][0] >= target), end)
        if next_cursor <= cursor: raise RuntimeError(f'La ventana no avanza: {example["doc_id"]}')
        cursor = next_cursor
    assert covered == represented
    return result

beto_windows_by_split, window_rows = {}, []
for split_name in ('train', 'validation'):
    windows=[]
    for di,(example,encoding) in enumerate(zip(bio_examples_by_split[split_name], beto_analysis_encodings[split_name])):
        windows.extend(build_windows(example, encoding, split_name, di))
    beto_windows_by_split[split_name]=windows
    for w in windows:
        assert len(w['input_ids']) <= BETO_WINDOW_LENGTH
        assert len(w['input_ids']) == len(w['labels']) == len(w['word_ids'])
        assert w['input_ids'][0] == beto_tokenizer.cls_token_id and w['input_ids'][-1] == beto_tokenizer.sep_token_id
        window_rows.append({'conjunto':split_name,'window_id':w['window_id'],'doc_id':w['doc_id'],
            'ventana':w['window_index'],'tokens_spacy':len(w['tokens']),'posiciones_totales':len(w['input_ids']),
            'posiciones_supervisadas':sum(y != BIO_IGNORE for y in w['labels'])})
window_table=pd.DataFrame(window_rows)
doc_table=(window_table.groupby(['conjunto','doc_id'],sort=False).agg(ventanas=('window_id','count'),
    posiciones_supervisadas=('posiciones_supervisadas','sum')).reset_index())
display(Markdown('### PASO 06 OK — Ventanas construidas'))
show_markdown_table(['Conjunto','Documentos','Ventanas','Mediana ventanas/documento','Máximo ventanas/documento','Máximo posiciones'],[
 [name,len(doc_table[doc_table.conjunto==name]),len(window_table[window_table.conjunto==name]),
  f'{doc_table[doc_table.conjunto==name].ventanas.median():.1f}',int(doc_table[doc_table.conjunto==name].ventanas.max()),int(window_table[window_table.conjunto==name].posiciones_totales.max())]
 for name in ('train','validation')])
show_markdown_table(['Verificación','Resultado'],[['Ventana máxima',int(window_table.posiciones_totales.max())],['Tokens spaCy sin cobertura',0],['Test procesado',0],['Solapamiento objetivo',BETO_OVERLAP_SUBTOKENS],['Etiquetado','Primer subtoken; continuaciones IGNORE']])
window_table.to_csv(WORK/'beto_windows.csv',index=False); doc_table.to_csv(WORK/'beto_document_windows.csv',index=False)
(WORK/'window_examples.json').write_text(json.dumps({k:v[:3] for k,v in beto_windows_by_split.items()},ensure_ascii=False,indent=2),encoding='utf-8')
REPORT['windows']={'window_length':BETO_WINDOW_LENGTH,'overlap_subtokens':BETO_OVERLAP_SUBTOKENS,'train_windows':len(beto_windows_by_split['train']),'validation_windows':len(beto_windows_by_split['validation']),'max_window_positions':int(window_table.posiciones_totales.max()),'test_processed':False,'word_boundary_aligned':True,'all_spacy_tokens_covered':True,'deduplication_required_at_evaluation':True}
REPORT['status']='VENTANAS_BETO_CONSTRUIDAS'; (WORK/'data_summary.json').write_text(json.dumps(REPORT,indent=2,ensure_ascii=False),encoding='utf-8')
display(Markdown('Las ventanas y sus etiquetas están listas para preparar el `DataCollator` del siguiente paso.'))

### PASO 06 OK — Ventanas construidas

| Conjunto | Documentos | Ventanas | Mediana ventanas/documento | Máximo ventanas/documento | Máximo posiciones |
| --- | --- | --- | --- | --- | --- |
| train | 600 | 934 | 1.0 | 4 | 512 |
| validation | 150 | 245 | 2.0 | 3 | 512 |

| Verificación | Resultado |
| --- | --- |
| Ventana máxima | 512 |
| Tokens spaCy sin cobertura | 0 |
| Test procesado | 0 |
| Solapamiento objetivo | 64 |
| Etiquetado | Primer subtoken; continuaciones IGNORE |

Las ventanas y sus etiquetas están listas para preparar el `DataCollator` del siguiente paso.

 ### Datos de las ventanas cradas

  La construcción de ventanas permitió transformar los documentos largos en entradas compatibles con BETO, utilizando un máximo de 512 posiciones y un
  solapamiento de 64 subtokens. Se generaron 934 ventanas para entrenamiento y 245 para validación a partir de 600 y 150 documentos, respectivamente.

  La mediana fue de una ventana por documento en entrenamiento y dos en validación, mientras que ningún documento requirió más de cuatro ventanas. Todas las
  ventanas respetaron el límite del modelo y se confirmó una cobertura completa de las entidades clínicas, sin tokens de entidad sin representación.

  Las etiquetas BIO se conservaron mediante la política de supervisar únicamente el primer subtoken y marcar las continuaciones como IGNORE. El conjunto de
  test no se proces

### PASO 07 — Datasets, collator y BERT para clasificación por token

Convertimos las ventanas en `datasets.Dataset` y usamos `DataCollatorForTokenClassification` para aplicar padding dinámico al tamaño máximo de cada lote. El collator rellena `input_ids`, `attention_mask` y `token_type_ids`; las posiciones nuevas de `labels` reciben `-100`, por lo que no afectan la pérdida.

Cargamos `AutoModelForTokenClassification` desde el mismo checkpoint que el tokenizador: `dccuchile/bert-base-spanish-wwm-cased`. La cabeza se inicializa para **11 etiquetas BIO**. En la siguiente etapa ajustaremos todos los pesos mediante fine-tuning; aquí solo comprobamos la carga, las dimensiones y el flujo de un lote. No procesamos test ni actualizamos parámetros.

In [12]:
# PASO 07 — Preparar Hugging Face y cargar BERT; aún sin entrenamiento.
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'datasets>=3,<5', 'accelerate>=1,<2'])

from datasets import Dataset, DatasetDict
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification
import torch

assert set(beto_windows_by_split) == {'train', 'validation'}
assert len(BIO_TAGS) == 11

MODEL_ID2LABEL = {i: tag for i, tag in enumerate(BIO_TAGS)}
MODEL_LABEL2ID = {tag: i for i, tag in MODEL_ID2LABEL.items()}
MODEL_NUM_LABELS = len(MODEL_ID2LABEL)

# Metadata fuera del Dataset evita que cadenas de auditoría lleguen al modelo o al collator.
hf_windows_metadata = {}
hf_datasets = {}
for split_name in ('train', 'validation'):
    windows = beto_windows_by_split[split_name]
    hf_windows_metadata[split_name] = [{key: window[key] for key in
        ('window_id', 'doc_id', 'split', 'window_index', 'start_word', 'end_word', 'offsets', 'tokens', 'entity_word_ids')}
        for window in windows]
    records = [{key: window[key] for key in ('input_ids', 'attention_mask', 'token_type_ids', 'labels')}
               for window in windows]
    hf_datasets[split_name] = Dataset.from_list(records)
hf_dataset = DatasetDict(hf_datasets)
assert set(hf_dataset) == {'train', 'validation'}
assert len(hf_dataset['train']) == len(beto_windows_by_split['train'])
assert len(hf_dataset['validation']) == len(beto_windows_by_split['validation'])
assert set(hf_dataset['train'].column_names) == {'input_ids', 'attention_mask', 'token_type_ids', 'labels'}

data_collator = DataCollatorForTokenClassification(tokenizer=beto_tokenizer, padding=True,
                                                     return_tensors='pt')

BERT_DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
bert_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=MODEL_NUM_LABELS, id2label=MODEL_ID2LABEL,
    label2id=MODEL_LABEL2ID, torch_dtype=torch.float32).to(BERT_DEVICE)
# Confirmar que la cabeza corresponde a nuestra tarea y que el encoder está disponible.
assert bert_model.config.num_labels == MODEL_NUM_LABELS
assert bert_model.config.id2label == MODEL_ID2LABEL
assert bert_model.config.label2id == MODEL_LABEL2ID
assert bert_model.config.max_position_embeddings >= BETO_WINDOW_LENGTH
assert hasattr(bert_model, 'base_model')

sample_batch = data_collator([hf_dataset['train'][i] for i in range(min(2, len(hf_dataset['train'])))])
sample_batch = {key: value.to(BERT_DEVICE) for key, value in sample_batch.items()}
with torch.no_grad():
    sample_output = bert_model(**sample_batch)
assert sample_output.logits.ndim == 3
assert sample_output.logits.shape[0] == len(sample_batch['input_ids'])
assert sample_output.logits.shape[1] == sample_batch['input_ids'].shape[1]
assert sample_output.logits.shape[2] == MODEL_NUM_LABELS
assert torch.isfinite(sample_output.logits).all()

parameter_total = sum(parameter.numel() for parameter in bert_model.parameters())
parameter_trainable = sum(parameter.numel() for parameter in bert_model.parameters() if parameter.requires_grad)
REPORT['bert'] = {'checkpoint': MODEL_CHECKPOINT, 'task': 'token_classification',
    'num_labels': MODEL_NUM_LABELS, 'id2label': MODEL_ID2LABEL, 'label2id': MODEL_LABEL2ID,
    'train_windows': len(hf_dataset['train']), 'validation_windows': len(hf_dataset['validation']),
    'datasets_columns': hf_dataset['train'].column_names, 'collator': 'DataCollatorForTokenClassification',
    'model_type': bert_model.config.model_type, 'parameter_total': parameter_total,
    'parameter_trainable_before_training': parameter_trainable, 'fine_tuning_planned': True,
    'weights_updated': False, 'test_processed': False,
    'sample_logits_shape': list(sample_output.logits.shape), 'device_for_smoke_test': str(BERT_DEVICE)}
(WORK / 'bert_setup.json').write_text(json.dumps(REPORT['bert'], indent=2, ensure_ascii=False), encoding='utf-8')
print('PASO 07 OK — BERT cargado sin entrenamiento.')
display(Markdown('### Verificación del modelo'))
show_markdown_table(['Propiedad', 'Resultado'], [
    ['Checkpoint', MODEL_CHECKPOINT], ['Tipo de tarea', 'Clasificación por token'],
    ['Etiquetas de salida', MODEL_NUM_LABELS], ['Modelo', bert_model.config.model_type],
    ['Parámetros totales', f'{parameter_total:,}'], ['Parámetros entrenables', f'{parameter_trainable:,}'],
    ['Forma de logits', str(tuple(sample_output.logits.shape))],
    ['Dataset train / validation', f"{len(hf_dataset['train'])} / {len(hf_dataset['validation'])} ventanas"],
    ['Test procesado', 'No'], ['Pesos actualizados', 'No'],
])
print('Datasets listos para Trainer o un bucle PyTorch en el PASO 08.')

`torch_dtype` is deprecated! Use `dtype` instead!
Some weights of BertForTokenClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PASO 07 OK — BERT cargado sin entrenamiento.


### Verificación del modelo

| Propiedad | Resultado |
| --- | --- |
| Checkpoint | dccuchile/bert-base-spanish-wwm-cased |
| Tipo de tarea | Clasificación por token |
| Etiquetas de salida | 11 |
| Modelo | bert |
| Parámetros totales | 109,268,747 |
| Parámetros entrenables | 109,268,747 |
| Forma de logits | (2, 511, 11) |
| Dataset train / validation | 934 / 245 ventanas |
| Test procesado | No |
| Pesos actualizados | No |

Datasets listos para Trainer o un bucle PyTorch en el PASO 08.


### Nota sobre la preparación de BERT
Las ventanas de entrenamiento y validación quedaron correctamente integradas mediante DataCollatorForTokenClassification, con 934 y 245 ventanas,
respectivamente. La salida del modelo presentó la forma (2, 511, 11), lo que confirma que genera 11 predicciones para cada posición de cada secuencia. La
longitud 511 se debe al padding dinámico del lote y no representa una reducción del límite máximo de 512 posiciones.

BETO está cargado con una cabeza de clasificación por token y 11 etiquetas BIO. Las ventanas y el collator se utilizarán en dos entrenamientos con los mismos datos y métricas: transfer learning con el encoder congelado, que ajusta únicamente la cabeza BIO, y fine-tuning completo, que ajusta todas las capas de BERT. El flujo de un lote fue verificado sin modificar los pesos. Test permanece reservado.

### PASO 08 — Transfer learning congelado frente a fine-tuning

Entrenamos dos modelos BERT independientes usando exactamente las mismas ventanas, etiquetas BIO, particiones y métricas:

| Variante | Capas BERT | Cabeza de clasificación | Tasa inicial |
|---|---|---|---:|
| `bert_frozen` | Congeladas | Entrenable | `1e-3` |
| `bert_finetuned` | Entrenables | Entrenable | `2e-5` |

En `bert_frozen`, BERT funciona como extractor de representaciones y solo se ajusta la capa que genera los 11 logits por token. En `bert_finetuned`, todo el checkpoint se adapta al dominio clínico. Usamos tasas diferentes porque la cabeza empieza aleatoria, mientras que un fine-tuning requiere actualizaciones pequeñas para conservar el conocimiento preentrenado.

La métrica principal es el F1 de entidades BIO calculado sobre las posiciones supervisadas de cada ventana; también reportamos precisión, recall y exactitud de token. El modelo con mejor F1 de validación se conserva automáticamente. Las ventanas solapadas pueden repetir una entidad, por lo que la evaluación final por documento deberá deduplicar mediante offsets. Test sigue reservado.

In [13]:
# PASO 08 — Dos estrategias de adaptación de BETO.
import copy
import time
import numpy as np
from transformers import Trainer, TrainingArguments, set_seed
from transformers.trainer_callback import EarlyStoppingCallback
# Métrica BIO exacta implementada localmente; no requiere seqeval.
def bio_entities(tags):
    entities, active_label, begin = set(), None, None
    for index, tag in enumerate(list(tags) + ['O']):
        if tag == 'O' or not tag:
            prefix, label = 'O', None
        else:
            prefix, label = tag.split('-', 1)
        if active_label is not None and not (prefix == 'I' and label == active_label):
            entities.add((begin, index, active_label))
            active_label, begin = None, None
        if prefix == 'B' or (prefix == 'I' and active_label is None):
            active_label, begin = label, index
    return entities


def bert_metrics(eval_prediction):
    predictions = eval_prediction.predictions[0] if isinstance(eval_prediction.predictions, tuple) else eval_prediction.predictions
    labels = eval_prediction.label_ids
    predicted_ids = np.argmax(predictions, axis=-1)
    true_sequences, predicted_sequences = [], []
    token_correct = token_total = 0
    true_entities = predicted_entities = true_positives = 0
    for prediction, reference in zip(predicted_ids, labels):
        true_sequence, predicted_sequence = [], []
        for pred_id, label_id in zip(prediction, reference):
            if int(label_id) == BIO_IGNORE:
                continue
            gold_tag, pred_tag = BIO_ID2LABEL[int(label_id)], BIO_ID2LABEL[int(pred_id)]
            true_sequence.append(gold_tag); predicted_sequence.append(pred_tag)
            token_correct += int(gold_tag == pred_tag); token_total += 1
        true_sequences.append(true_sequence); predicted_sequences.append(predicted_sequence)
        gold_entities, pred_entities = bio_entities(true_sequence), bio_entities(predicted_sequence)
        true_entities += len(gold_entities); predicted_entities += len(pred_entities)
        true_positives += len(gold_entities & pred_entities)
    precision = true_positives / predicted_entities if predicted_entities else 0.0
    recall = true_positives / true_entities if true_entities else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1,
            'token_accuracy': token_correct / token_total if token_total else 0.0}


# Una prueba rápida comprueba que la métrica tolera padding y posiciones IGNORE.
_metric_sample = type('EvalPrediction', (), {
    'predictions': np.zeros((1, 3, MODEL_NUM_LABELS)),
    'label_ids': np.array([[BIO_IGNORE, MODEL_LABEL2ID['O'], MODEL_LABEL2ID['B-DISEASE']]])})
_metric_sample.predictions[0, 1, MODEL_LABEL2ID['O']] = 1
_metric_sample.predictions[0, 2, MODEL_LABEL2ID['B-DISEASE']] = 1
assert bert_metrics(_metric_sample)['f1'] == 1.0

BERT_TRAINING_SEED = 42
BERT_EPOCHS = 8
BERT_PATIENCE = 2
BERT_BATCH_SIZE = 8
BERT_WEIGHT_DECAY = 0.01
BERT_OUTPUT_ROOT = WORK / 'bert_training'
BERT_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def make_training_arguments(output_dir, learning_rate, training_seed=BERT_TRAINING_SEED):
    common = dict(output_dir=str(output_dir), num_train_epochs=BERT_EPOCHS,
        learning_rate=learning_rate, per_device_train_batch_size=BERT_BATCH_SIZE,
        per_device_eval_batch_size=BERT_BATCH_SIZE, weight_decay=BERT_WEIGHT_DECAY,
        save_strategy='epoch', logging_strategy='epoch', load_best_model_at_end=True,
        metric_for_best_model='eval_f1', greater_is_better=True, save_total_limit=2,
        report_to='none', seed=training_seed, data_seed=training_seed,
        fp16=torch.cuda.is_available(), remove_unused_columns=True)
    try:
        return TrainingArguments(eval_strategy='epoch', **common)
    except TypeError:
        return TrainingArguments(evaluation_strategy='epoch', **common)


def train_bert_variant(name, freeze_base, learning_rate, training_seed=None, output_root=None):
    run_seed = BERT_TRAINING_SEED if training_seed is None else int(training_seed)
    run_root = BERT_OUTPUT_ROOT if output_root is None else output_root
    set_seed(run_seed)
    output_dir = run_root / name
    output_dir.mkdir(parents=True, exist_ok=True)
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_CHECKPOINT, num_labels=MODEL_NUM_LABELS, id2label=MODEL_ID2LABEL,
        label2id=MODEL_LABEL2ID, torch_dtype=torch.float32)
    if freeze_base:
        for parameter in model.base_model.parameters():
            parameter.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    args = make_training_arguments(output_dir, learning_rate, run_seed)
    trainer = Trainer(model=model, args=args, train_dataset=hf_dataset['train'],
        eval_dataset=hf_dataset['validation'], data_collator=data_collator,
        compute_metrics=bert_metrics, processing_class=beto_tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=BERT_PATIENCE)])
    start = time.perf_counter()
    trainer.train()
    elapsed = time.perf_counter() - start
    metrics = trainer.evaluate(eval_dataset=hf_dataset['validation'])
    required_metrics = {'eval_precision', 'eval_recall', 'eval_f1', 'eval_token_accuracy'}
    assert required_metrics <= metrics.keys(), f'Métricas de validación ausentes: {required_metrics - metrics.keys()}'
    trainer.save_model(str(output_dir / 'best_model'))
    result = {'variant': name, 'freeze_base': freeze_base, 'seed': run_seed,
        'learning_rate': learning_rate, 'epochs': BERT_EPOCHS, 'patience': BERT_PATIENCE,
        'parameters_total': total, 'parameters_trainable': trainable,
        'train_seconds': elapsed, 'validation_metrics': metrics,
        'training_device_count': torch.cuda.device_count(),
        'best_checkpoint': str(output_dir / 'best_model'), 'test_processed': False,
        'dataset_windows': {'train': len(hf_dataset['train']), 'validation': len(hf_dataset['validation'])}}
    (output_dir / 'result.json').write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')
    return result


BERT_TRAINING_CONFIG = {'seed': BERT_TRAINING_SEED, 'epochs': BERT_EPOCHS,
    'patience': BERT_PATIENCE, 'batch_size': BERT_BATCH_SIZE,
    'same_windows': True, 'test_processed': False}
bert_results = {}
for name, frozen, lr in [('bert_frozen', True, 1e-3), ('bert_finetuned', False, 2e-5)]:
    print(f'Iniciando {name}: base congelada={frozen}, lr={lr}', flush=True)
    bert_results[name] = train_bert_variant(name, frozen, lr)

bert_comparison = pd.DataFrame([{
    'modelo': name, 'base_congelada': result['freeze_base'],
    'parametros_totales': result['parameters_total'],
    'parametros_entrenables': result['parameters_trainable'],
    'precision_pct': 100 * result['validation_metrics']['eval_precision'],
    'recall_pct': 100 * result['validation_metrics']['eval_recall'],
    'f1_pct': 100 * result['validation_metrics']['eval_f1'],
    'exactitud_token_pct': 100 * result['validation_metrics']['eval_token_accuracy'],
    'tiempo_entrenamiento_s': result['train_seconds']}
    for name, result in bert_results.items()])
bert_comparison.to_csv(BERT_OUTPUT_ROOT / 'bert_frozen_vs_finetuned.csv', index=False)
REPORT['bert_training'] = {'config': BERT_TRAINING_CONFIG, 'results': bert_results,
    'comparison_csv': str(BERT_OUTPUT_ROOT / 'bert_frozen_vs_finetuned.csv'),
    'metric': 'F1 BIO exacto local sobre posiciones supervisadas por ventana',
    'test_processed': False}
(WORK / 'data_summary.json').write_text(json.dumps(REPORT, indent=2, ensure_ascii=False), encoding='utf-8')
display(Markdown('### PASO 08 OK — Comparación de adaptación de BERT'))
display(bert_comparison.style.format({
    'precision_pct':'{:.2f}', 'recall_pct':'{:.2f}', 'f1_pct':'{:.2f}',
    'exactitud_token_pct':'{:.2f}', 'tiempo_entrenamiento_s':'{:.1f}'}))
print('Resultados guardados en:', BERT_OUTPUT_ROOT / 'bert_frozen_vs_finetuned.csv')

Iniciando bert_frozen: base congelada=True, lr=0.001


Some weights of BertForTokenClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Token Accuracy
1,1.031200,0.656585,0.238894,0.321939,0.274268,0.774791
2,0.641500,0.582975,0.281897,0.407313,0.333194,0.797120
3,0.597600,0.558540,0.299859,0.435204,0.355071,0.804159
4,0.579400,0.543669,0.306262,0.450850,0.364750,0.807824
5,0.566200,0.535376,0.307945,0.462755,0.369802,0.810569
6,0.560700,0.530503,0.316561,0.466497,0.377174,0.812135
7,0.557600,0.527805,0.315964,0.470238,0.377965,0.812441
8,0.553300,0.526958,0.317333,0.471088,0.379218,0.812700


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

Iniciando bert_finetuned: base congelada=False, lr=2e-05


...

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Token Accuracy
1,0.870800,0.499417,0.388752,0.456122,0.419751,0.825002
2,0.434300,0.403983,0.480197,0.595918,0.531836,0.860457
3,0.349400,0.381258,0.533720,0.625850,0.576125,0.870871
4,0.295700,0.366323,0.562996,0.652041,0.604255,0.878136
5,0.261700,0.376785,0.564039,0.662075,0.609138,0.878346
6,0.232900,0.376070,0.580161,0.663435,0.619010,0.881462
7,0.216600,0.379176,0.579048,0.670238,0.621315,0.881075
8,0.203400,0.386063,0.583913,0.670408,0.624179,0.881349


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

### PASO 08 OK — Comparación de adaptación de BERT

,modelo,base_congelada,parametros_totales,parametros_entrenables,precision_pct,recall_pct,f1_pct,exactitud_token_pct,tiempo_entrenamiento_s
0,bert_frozen,True,109268747,8459,31.73,47.11,37.92,81.27,211.3
1,bert_finetuned,False,109268747,109268747,58.39,67.04,62.42,88.13,476.7


Resultados guardados en: /kaggle/working/spaccc_bert/bert_training/bert_frozen_vs_finetuned.csv


 ### Resultados paso 8

  El ajuste fino de BETO obtuvo mejores resultados que mantener congelada la base. El modelo bert_finetuned alcanzó un F1 de 62,42%, frente a 37,92% de
  bert_frozen, con mejoras también en precisión, recall y exactitud por token. Esto muestra que las representaciones preentrenadas necesitan adaptarse al
  vocabulario y a las entidades específicas del corpus. El modelo congelado requiere menos parámetros entrenables y menor tiempo, pero su capacidad de
  adaptación es limitada. Estos resultados corresponden al conjunto de validación

### PASO 09 — Repetición multisemilla y estabilidad

Repetimos las dos estrategias con las semillas `42`, `123` y `2026`. La semilla solo cambia la aleatoriedad de entrenamiento; documentos, ventanas, etiquetas, hiperparámetros y métrica permanecen fijos. Cada ejecución carga un checkpoint BERT nuevo, para evitar que una variante herede pesos de otra.

Este paso produce seis ejecuciones independientes y calcula media, desviación estándar y diferencias pareadas. La media describe el comportamiento esperado; la desviación muestra sensibilidad a la semilla. Elegiremos la estrategia por el F1 medio de validación. No seleccionaremos una semilla por ser la mejor y no procesaremos test.

In [14]:
# PASO 09 — Seis ejecuciones: 2 variantes × 3 semillas.
BERT_SEEDS = [42, 123, 2026]
BERT_MULTISEED_ROOT = WORK / 'bert_training_multiseed'
BERT_MULTISEED_ROOT.mkdir(parents=True, exist_ok=True)
assert BERT_EPOCHS >= 8 and BERT_PATIENCE >= 2

multiseed_results = {}
for seed in BERT_SEEDS:
    multiseed_results[str(seed)] = {}
    for name, frozen, lr in [('bert_frozen', True, 1e-3), ('bert_finetuned', False, 2e-5)]:
        print(f'Iniciando {name}, semilla {seed}, épocas máximas={BERT_EPOCHS}', flush=True)
        multiseed_results[str(seed)][name] = train_bert_variant(
            name, frozen, lr, training_seed=seed,
            output_root=BERT_MULTISEED_ROOT / f'seed_{seed}')

multiseed_rows = []
for seed, pair in multiseed_results.items():
    for name, result in pair.items():
        metrics = result['validation_metrics']
        multiseed_rows.append({'semilla': int(seed), 'modelo': name,
            'base_congelada': result['freeze_base'],
            'precision_pct': 100 * metrics['eval_precision'],
            'recall_pct': 100 * metrics['eval_recall'],
            'f1_pct': 100 * metrics['eval_f1'],
            'exactitud_token_pct': 100 * metrics['eval_token_accuracy'],
            'parametros_entrenables': result['parameters_trainable'],
            'tiempo_entrenamiento_s': result['train_seconds']})
multiseed_table = pd.DataFrame(multiseed_rows)
multiseed_summary = (multiseed_table.groupby('modelo', sort=False)
    .agg(n_semillas=('semilla','count'), precision_media_pct=('precision_pct','mean'),
         precision_std_pct=('precision_pct','std'), recall_media_pct=('recall_pct','mean'),
         recall_std_pct=('recall_pct','std'), f1_media_pct=('f1_pct','mean'),
         f1_std_pct=('f1_pct','std'), tiempo_medio_s=('tiempo_entrenamiento_s','mean'))
    .reset_index())
paired = multiseed_table.pivot(index='semilla', columns='modelo', values='f1_pct').reset_index()
paired['diferencia_f1_pp_finetuned_menos_frozen'] = paired['bert_finetuned'] - paired['bert_frozen']
assert len(multiseed_table) == 6 and len(paired) == 3
assert (multiseed_table.groupby('semilla').size() == 2).all()
BERT_MULTISEED_ROOT.mkdir(parents=True, exist_ok=True)
multiseed_table.to_csv(BERT_MULTISEED_ROOT / 'bert_multiseed_por_semilla.csv', index=False)
multiseed_summary.to_csv(BERT_MULTISEED_ROOT / 'bert_multiseed_resumen.csv', index=False)
paired.to_csv(BERT_MULTISEED_ROOT / 'bert_multiseed_pareada.csv', index=False)
REPORT['bert_multiseed'] = {'seeds': BERT_SEEDS, 'config': {'epochs': BERT_EPOCHS, 'patience': BERT_PATIENCE,
    'batch_size': BERT_BATCH_SIZE, 'same_windows': True}, 'per_seed': multiseed_rows,
    'summary': multiseed_summary.to_dict('records'), 'paired': paired.to_dict('records'),
    'test_processed': False, 'selection_rule': 'Mayor F1 medio de validación; no se escoge semilla por su resultado.'}
(WORK / 'data_summary.json').write_text(json.dumps(REPORT, indent=2, ensure_ascii=False), encoding='utf-8')
display(Markdown('### PASO 09 OK — Resultados multisemilla'))
display(multiseed_table.style.format({'precision_pct':'{:.2f}','recall_pct':'{:.2f}','f1_pct':'{:.2f}',
    'exactitud_token_pct':'{:.2f}','tiempo_entrenamiento_s':'{:.1f}'}))
display(multiseed_summary.style.format({'precision_media_pct':'{:.2f}','precision_std_pct':'{:.2f}',
    'recall_media_pct':'{:.2f}','recall_std_pct':'{:.2f}','f1_media_pct':'{:.2f}',
    'f1_std_pct':'{:.2f}','tiempo_medio_s':'{:.1f}'}))
display(paired.style.format({'bert_frozen':'{:.2f}','bert_finetuned':'{:.2f}',
    'diferencia_f1_pp_finetuned_menos_frozen':'{:+.2f}'}))
print('Artefactos guardados en:', BERT_MULTISEED_ROOT)

Iniciando bert_frozen, semilla 42, épocas máximas=8


Some weights of BertForTokenClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Token Accuracy
1,1.031200,0.656585,0.238894,0.321939,0.274268,0.774791
2,0.641500,0.582975,0.281897,0.407313,0.333194,0.797120
3,0.597600,0.558540,0.299859,0.435204,0.355071,0.804159
4,0.579400,0.543669,0.306262,0.450850,0.364750,0.807824
5,0.566200,0.535376,0.307945,0.462755,0.369802,0.810569
6,0.560700,0.530503,0.316561,0.466497,0.377174,0.812135
7,0.557600,0.527805,0.315964,0.470238,0.377965,0.812441
8,0.553300,0.526958,0.317333,0.471088,0.379218,0.812700


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

Iniciando bert_finetuned, semilla 42, épocas máximas=8


...

### PASO 09 OK — Resultados multisemilla

,semilla,modelo,base_congelada,precision_pct,recall_pct,f1_pct,exactitud_token_pct,parametros_entrenables,tiempo_entrenamiento_s
0,42,bert_frozen,True,31.73,47.11,37.92,81.27,8459,207.2
1,42,bert_finetuned,False,58.39,67.04,62.42,88.13,109268747,476.8
2,123,bert_frozen,True,31.59,46.92,37.76,81.28,8459,209.2
3,123,bert_finetuned,False,58.96,67.89,63.11,88.25,109268747,480.1
4,2026,bert_frozen,True,31.79,46.90,37.90,81.33,8459,207.3
5,2026,bert_finetuned,False,58.36,67.31,62.52,88.10,109268747,476.3


,modelo,n_semillas,precision_media_pct,precision_std_pct,recall_media_pct,recall_std_pct,f1_media_pct,f1_std_pct,tiempo_medio_s
0,bert_frozen,3,31.70,0.10,46.98,0.11,37.86,0.09,207.9
1,bert_finetuned,3,58.57,0.34,67.41,0.43,62.68,0.37,477.7


modelo,semilla,bert_finetuned,bert_frozen,diferencia_f1_pp_finetuned_menos_frozen
0,42,62.42,37.92,+24.50
1,123,63.11,37.76,+25.35
2,2026,62.52,37.90,+24.62


Artefactos guardados en: /kaggle/working/spaccc_bert/bert_training_multiseed


### Conclusiones del PASO 09

  La evaluación con tres semillas confirma que el modelo bert_finetuned supera ampliamente al modelo bert_frozen de forma consistente.

  El modelo con ajuste fino obtuvo un F1 medio de 62,68%, frente a 37,86% del modelo congelado. La mejora promedio fue de 24,82 puntos porcentuales, con
  diferencias similares en las tres semillas: entre 24,50 y 25,35 puntos.

  Los resultados presentan poca variabilidad. La desviación estándar del F1 fue de 0,37 puntos para bert_finetuned y de 0,09 puntos para bert_frozen. Esto
  indica que la ventaja del ajuste fino no depende de una semilla específica.

  La semilla 123 produjo el mejor resultado individual para bert_finetuned, con un F1 de 63,11%. Sin embargo, la diferencia respecto a las otras semillas es
  pequeña, por lo que no se debe concluir que sea una “mejor semilla” universal.

  El modelo congelado entrenó aproximadamente en 208 segundos, mientras que el modelo con ajuste fino necesitó cerca de 478 segundos y actualizó todos los
  parámetros de BETO. Por tanto, bert_finetuned ofrece un rendimiento muy superior a cambio de un mayor costo computacional.

### PASO 10 — Evaluación final sobre el conjunto de prueba

Seleccionamos la variante con mayor F1 medio de validación en el PASO 09. Después procesamos el conjunto de prueba con la misma tokenización, alineación BIO y ventanas de 512 posiciones con solapamiento de 64 subtokens. El conjunto de prueba se utiliza una sola vez y las predicciones repetidas por el solapamiento se deduplican por documento y token spaCy antes de calcular las métricas.

In [15]:
# PASO 10 — Evaluación final, una sola vez, sobre test.
# El PASO 09 ya terminó el entrenamiento. Aquí solo leemos sus artefactos.
BERT_MULTISEED_ROOT = globals().get('BERT_MULTISEED_ROOT', WORK / 'bert_training_multiseed')
if 'multiseed_table' in globals() and 'multiseed_results' in globals():
    selected_seed = int(multiseed_table[multiseed_table['modelo'] == 'bert_finetuned']
        .sort_values('f1_pct', ascending=False).iloc[0]['semilla'])
    selected_result = multiseed_results[str(selected_seed)]['bert_finetuned']
else:
    summary_path = BERT_MULTISEED_ROOT / 'bert_multiseed_por_semilla.csv'
    assert summary_path.exists(), 'Ejecuta el PASO 09 para crear los resultados multisemilla.'
    saved_table = pd.read_csv(summary_path)
    selected_seed = int(saved_table[saved_table['modelo'] == 'bert_finetuned']
        .sort_values('f1_pct', ascending=False).iloc[0]['semilla'])
    result_path = BERT_MULTISEED_ROOT / f'seed_{selected_seed}' / 'bert_finetuned' / 'result.json'
    assert result_path.exists(), f'No existe el resultado de la semilla seleccionada: {result_path}'
    selected_result = json.loads(result_path.read_text(encoding='utf-8'))
selected_checkpoint = Path(selected_result['best_checkpoint'])
assert selected_checkpoint.exists(), f'No existe el checkpoint seleccionado: {selected_checkpoint}'
print(f'Checkpoint reutilizado del PASO 09: semilla {selected_seed}')

# Esta función también se define aquí para permitir evaluar sin ejecutar el entrenamiento del PASO 08.
if 'bio_entities' not in globals():
    def bio_entities(tags):
        entities, active_label, begin = set(), None, None
        for index, tag in enumerate(list(tags) + ['O']):
            if tag == 'O' or not tag:
                prefix, label = 'O', None
            else:
                prefix, label = tag.split('-', 1)
            if active_label is not None and not (prefix == 'I' and label == active_label):
                entities.add((begin, index, active_label)); active_label, begin = None, None
            if prefix == 'B' or (prefix == 'I' and active_label is None):
                active_label, begin = label, index

# Repetimos en test la misma inspección BIO y la misma construcción de ventanas.
_, test_bio_stats, test_examples = inspect_bio(
    test_ids, annotations_by_split['test'], texts, bio_nlp, return_examples=True)
test_encodings = []
test_windows = []
for document_index, example in enumerate(test_examples):
    encoding = beto_tokenizer(example['tokens'], is_split_into_words=True,
        add_special_tokens=True, truncation=False, padding=False,
        return_special_tokens_mask=True, verbose=False)
    encoding['labels'] = align_first_subtoken(encoding, example['tags'])
    test_encodings.append(encoding)
    test_windows.extend(build_windows(example, encoding, 'test', document_index))
assert test_windows and all(len(w['input_ids']) <= BETO_WINDOW_LENGTH for w in test_windows)
assert all(len(w['input_ids']) == len(w['labels']) == len(w['word_ids']) for w in test_windows)

test_metadata = [{key: window[key] for key in
    ('window_id', 'doc_id', 'window_index', 'word_ids', 'tokens', 'entity_word_ids')}
    for window in test_windows]
test_records = [{key: window[key] for key in ('input_ids', 'attention_mask', 'token_type_ids', 'labels')}
                for window in test_windows]
test_dataset = Dataset.from_list(test_records)

# Se carga el mejor modelo ya entrenado; no se actualizan pesos durante la evaluación.
final_model = AutoModelForTokenClassification.from_pretrained(selected_checkpoint)
final_args = TrainingArguments(output_dir=str(BERT_MULTISEED_ROOT / 'final_test_eval'),
    per_device_eval_batch_size=BERT_BATCH_SIZE, report_to='none',
    fp16=torch.cuda.is_available(), remove_unused_columns=True)
final_trainer = Trainer(model=final_model, args=final_args,
    data_collator=data_collator, processing_class=beto_tokenizer)
prediction_output = final_trainer.predict(test_dataset)
predicted_ids = np.argmax(prediction_output.predictions[0]
    if isinstance(prediction_output.predictions, tuple) else prediction_output.predictions, axis=-1)

# Dedupliquemos posiciones que aparecen en dos ventanas por el solapamiento.
by_document = {example['doc_id']: {} for example in test_examples}
for row_index, metadata in enumerate(test_metadata):
    for position, (word_id, label_id, predicted_id) in enumerate(
            zip(metadata['word_ids'], test_records[row_index]['labels'], predicted_ids[row_index])):
        if word_id is None or int(label_id) == BIO_IGNORE:
            continue
        key = int(word_id)
        by_document[metadata['doc_id']].setdefault(key, (
            BIO_ID2LABEL[int(label_id)], BIO_ID2LABEL[int(predicted_id)]))

true_entities = predicted_entities = true_positives = token_correct = token_total = 0
for example in test_examples:
    gold_sequence, predicted_sequence = [], []
    for word_id, gold_tag in enumerate(example['tags']):
        if gold_tag == BIO_IGNORE:
            continue
        pair = by_document[example['doc_id']].get(word_id)
        if pair is None:
            continue
        gold_sequence.append(pair[0]); predicted_sequence.append(pair[1])
        token_correct += int(pair[0] == pair[1]); token_total += 1
    gold_set, predicted_set = bio_entities(gold_sequence), bio_entities(predicted_sequence)
    true_entities += len(gold_set); predicted_entities += len(predicted_set)
    true_positives += len(gold_set & predicted_set)
final_precision = true_positives / predicted_entities if predicted_entities else 0.0
final_recall = true_positives / true_entities if true_entities else 0.0
final_f1 = (2 * final_precision * final_recall / (final_precision + final_recall)
            if final_precision + final_recall else 0.0)
final_token_accuracy = token_correct / token_total if token_total else 0.0
final_metrics = {'precision': final_precision, 'recall': final_recall, 'f1': final_f1,
    'token_accuracy': final_token_accuracy, 'true_entities': true_entities,
    'predicted_entities': predicted_entities, 'true_positives': true_positives,
    'supervised_tokens_evaluated': token_total}
final_result = {'model': 'bert_finetuned', 'seed': selected_seed,
    'checkpoint': str(selected_checkpoint), 'test_windows': len(test_windows),
    'test_documents': len(test_examples), 'test_bio_audit': dict(test_bio_stats),
    'metrics': final_metrics, 'test_processed': True,
    'deduplication': 'first prediction per (doc_id, spaCy word_id) across overlapping windows'}
BERT_FINAL_ROOT = WORK / 'bert_final_test'
BERT_FINAL_ROOT.mkdir(parents=True, exist_ok=True)
(BERT_FINAL_ROOT / 'final_test_result.json').write_text(
    json.dumps(final_result, indent=2, ensure_ascii=False), encoding='utf-8')
REPORT['bert_final_test'] = final_result
(WORK / 'data_summary.json').write_text(json.dumps(REPORT, indent=2, ensure_ascii=False), encoding='utf-8')
final_table = pd.DataFrame([{'modelo': 'bert_finetuned', 'semilla': selected_seed,
    'precision_pct': 100 * final_precision, 'recall_pct': 100 * final_recall,
    'f1_pct': 100 * final_f1, 'exactitud_token_pct': 100 * final_token_accuracy,
    'documentos_test': len(test_examples), 'ventanas_test': len(test_windows)}])
display(Markdown('### PASO 10 OK — Evaluación final en test'))
display(final_table.style.format({'precision_pct':'{:.2f}', 'recall_pct':'{:.2f}',
    'f1_pct':'{:.2f}', 'exactitud_token_pct':'{:.2f}'}))
print('Checkpoint evaluado:', selected_checkpoint)
print('Resultado guardado en:', BERT_FINAL_ROOT / 'final_test_result.json')


The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Checkpoint reutilizado del PASO 09: semilla 123


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


### PASO 10 OK — Evaluación final en test

,modelo,semilla,precision_pct,recall_pct,f1_pct,exactitud_token_pct,documentos_test,ventanas_test
0,bert_finetuned,123,58.00,66.87,62.12,87.71,250,392


Checkpoint evaluado: /kaggle/working/spaccc_bert/bert_training_multiseed/seed_123/bert_finetuned/best_model
Resultado guardado en: /kaggle/working/spaccc_bert/bert_final_test/final_test_result.json


### Conclusiones del PASO 10

  La evaluación final sobre los 250 documentos de prueba se realizó con bert_finetuned y la semilla 123, seleccionados a partir del mejor F1 de validación. El
  modelo alcanzó:

  - Precisión: 58,00%
  - Recall: 66,87%
  - F1: 62,12%
  - Exactitud por token: 87,71%

  El resultado de test es muy cercano al obtenido en validación: 63,11% de F1, con una disminución de aproximadamente 0,99 puntos porcentuales. Esta
  diferencia es pequeña y sugiere que el modelo generaliza adecuadamente y no presenta un sobreajuste importante.

  El modelo procesó los 250 documentos mediante 392 ventanas, aplicando la misma tokenización, alineación BIO y deduplicación de tokens solapados utilizadas
  durante el desarrollo.

  Comparado con el modelo LSTM del taller anterior, cuyo F1 fue aproximadamente 38,96%, BETO con ajuste fino mejora el resultado en cerca de 23 puntos
  porcentuales. También supera ampliamente al Transformer entrenado desde cero.

  La exactitud por token debe interpretarse con cuidado porque la etiqueta O representa la mayoría de los tokens. Por ello, el F1 de entidades es la métrica
  principal para evaluar este problema.

### Conclusión general

  Los resultados de los tres talleres muestran que el tamaño y las características del corpus influyen directamente en la capacidad de los modelos para
  aprender la tarea de reconocimiento de entidades clínicas. El conjunto disponible contiene 750 documentos para desarrollo, de los cuales 600 se utilizaron
  para entrenamiento y 150 para validación, además de 250 documentos reservados para la evaluación final. Aunque el corpus contiene muchas anotaciones, la
  mayoría de los tokens pertenece a la clase O, lo que genera un problema de desbalance. También existen entidades solapadas, anotaciones no alineadas y
  documentos largos que deben dividirse en ventanas.

  El modelo LSTM obtuvo un F1 aproximado de 38,96%, mientras que el Transformer entrenado desde cero alcanzó cerca de 26,69%. Por tanto, en este corpus el
  Transformer construido desde cero no superó al modelo recurrente. Esta diferencia puede explicarse porque los Transformers tienen una mayor cantidad de
  parámetros y requieren más datos para aprender desde cero representaciones lingüísticas, relaciones contextuales y patrones propios del dominio clínico. Con
  solo 600 documentos de entrenamiento, el corpus resulta pequeño para aprovechar completamente la capacidad de un Transformer inicializado aleatoriamente.

  El uso de BETO, un modelo BERT preentrenado en español, produjo resultados considerablemente mejores. El modelo con el encoder congelado obtuvo un F1 medio
  de 37,86%, similar al LSTM, porque solo se entrenó una pequeña cabeza de clasificación. En cambio, bert_finetuned, que permitió actualizar todos los
  parámetros del modelo, alcanzó un F1 medio de 62,68% en validación. La mejora se mantuvo con las tres semillas evaluadas, con una desviación estándar de
  apenas 0,37 puntos porcentuales, lo que demuestra que no depende de una única inicialización aleatoria.

  La evaluación final sobre los 250 documentos de prueba produjo una precisión de 58,00%, un recall de 66,87% y un F1 de 62,12%. Este valor es muy cercano al
  obtenido en validación, lo que indica una buena capacidad de generalización y ausencia de un sobreajuste significativo. Además, el resultado supera en
  aproximadamente 23 puntos porcentuales al LSTM y en más de 35 puntos porcentuales al Transformer entrenado desde cero.

  En consecuencia, los experimentos evidencian que, para este corpus clínico relativamente pequeño, entrenar un Transformer desde cero no es la alternativa
  más adecuada. El preentrenamiento aporta conocimiento lingüístico previamente aprendido y permite que el modelo utilice el corpus disponible para
  especializarse en las entidades clínicas. El fine-tuning de BETO ofrece el mejor equilibrio entre aprovechamiento de los datos, calidad de predicción y
  capacidad de generalización, aunque requiere mayor tiempo y recursos computacionales que el modelo congelado.